# Projeto Integrador · Big Data · Mercado Imobiliário de São Paulo

<a href="https://colab.research.google.com/github/Real-Estate-VMs/real-estate/blob/main/notebooks/PI_BigData.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

**Empresa (fictícia):** JLeandro Imóveis · **Disciplina:** Big Data · **Módulo:** Data Science, 2º semestre de 2026

Este notebook segue as **6 etapas** do documento de orientações da disciplina:

| Etapa | Seção |
|---|---|
| 1. Definição do problema e da base | [1](#1) |
| 2. EDA | [2](#2) |
| 3. Armazenamento, ETL e OLAP | [3](#3) |
| 4. Data Mining e Machine Learning | [4](#4) |
| 5. Processamento distribuído (Spark / Hadoop) | [5](#5) |
| 6. Análise final e insights | [6](#6) |

**Como executar.** O notebook roda do início ao fim em dois modos, escolhidos automaticamente:

* **Modo `github`** (Google Colab ou qualquer máquina): lê os arquivos públicos do repositório no GitHub e monta o Data Warehouse **em memória** com DuckDB, usando as mesmas regras do ETL real.
* **Modo `dw`** (Colab conectado à VM `bigdata` por túnel SSH): lê o **Data Warehouse real** em PostgreSQL, alimentado automaticamente pelo pipeline.

As consultas SQL de OLAP são as mesmas nos dois modos.

In [1]:
import os
import sys

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    # O Colab já traz pandas, scikit-learn, matplotlib e seaborn
    get_ipython().run_line_magic("pip", "install -q duckdb==1.1.3 pyspark==3.5.3")
print("Rodando no Google Colab" if EM_COLAB else "Rodando fora do Colab (ex.: VM bigdata)")

Rodando fora do Colab (ex.: VM bigdata)


In [2]:
import ast
import json
import tempfile
import urllib.request
import warnings
from concurrent.futures import ThreadPoolExecutor
from decimal import Decimal

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

REPO = "Real-Estate-VMs/real-estate"
RAW_BASE = f"https://raw.githubusercontent.com/{REPO}/main"
PASTA_DADOS = os.path.join(tempfile.gettempdir(), "pi_bigdata")
# Na VM bigdata, REPO_LOCAL aponta para o clone que o ETL mantém atualizado: nada é baixado.
REPO_LOCAL = os.getenv("REPO_LOCAL") if os.path.isdir(os.getenv("REPO_LOCAL", "")) else None
os.makedirs(os.path.join(PASTA_DADOS, "raw"), exist_ok=True)
os.makedirs(os.path.join(PASTA_DADOS, "processed"), exist_ok=True)


def dw_disponivel():
    """True quando o notebook roda na VM bigdata com acesso ao PostgreSQL."""
    dsn = os.getenv("DW_DSN")
    if not dsn:
        return False
    try:
        import psycopg
        with psycopg.connect(dsn, connect_timeout=3):
            return True
    except Exception:
        return False


MODO = os.getenv("MODO_NOTEBOOK") or ("dw" if dw_disponivel() else "github")
print(f"Modo de execução: {MODO}")

Modo de execução: dw


In [3]:
def _arvore_repo():
    """Lista todos os arquivos do repositório numa única chamada à API do GitHub.

    Diferente de listar uma pasta (limite de 1000 itens), a árvore completa não corta os lotes antigos.
    """
    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/git/trees/main?recursive=1",
        headers={"Accept": "application/vnd.github+json", "User-Agent": "pi-bigdata-notebook"},
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        resposta = json.load(resp)
    if resposta.get("truncated"):
        print("[AVISO] a árvore do repositório veio truncada; alguns arquivos podem faltar")
    return [i["path"] for i in resposta["tree"] if i["type"] == "blob"]


ARVORE = None if REPO_LOCAL else _arvore_repo()


def listar_arquivos(pasta, extensao):
    """Lista os arquivos de uma pasta do repositório (clone local na VM, árvore do GitHub no Colab)."""
    if REPO_LOCAL:
        return sorted(f for f in os.listdir(os.path.join(REPO_LOCAL, pasta)) if f.endswith(extensao))
    return sorted(p.rsplit("/", 1)[1] for p in ARVORE if p.startswith(pasta + "/") and p.endswith(extensao))


def baixar(pasta, nome):
    """Devolve o caminho local de um arquivo do repositório, baixando-o uma vez se necessário."""
    if REPO_LOCAL:
        return os.path.join(REPO_LOCAL, pasta, nome)
    destino = os.path.join(PASTA_DADOS, pasta.split("/")[-1], nome)
    if not os.path.exists(destino):
        temporario = destino + ".part"  # evita deixar um arquivo pela metade se o download falhar
        urllib.request.urlretrieve(f"{RAW_BASE}/{pasta}/{nome}", temporario)
        os.replace(temporario, destino)
    return destino


def baixar_todos(pasta, nomes):
    """Baixa vários arquivos em paralelo (o gargalo é a latência, não a banda)."""
    with ThreadPoolExecutor(max_workers=16) as pool:
        return list(pool.map(lambda n: baixar(pasta, n), nomes))


arquivos_raw = listar_arquivos("data/raw", ".csv")
arquivos_processed = listar_arquivos("data/processed", ".parquet")
baixar_todos("data/raw", arquivos_raw)
baixar_todos("data/processed", arquivos_processed)
print(f"{len(arquivos_raw)} CSVs brutos e {len(arquivos_processed)} parquets tratados no repositório"
      + (" (lidos do clone local)" if REPO_LOCAL else " (baixados do GitHub)"))

11 CSVs brutos e 11 parquets tratados no repositório (lidos do clone local)


<a id="1"></a>
## 1. Definição do problema e da base

### Pergunta de análise
> **Quais fatores determinam o preço de um imóvel em São Paulo e quanto vale um imóvel antes de ser anunciado?**

Para a JLeandro Imóveis, responder isso significa precificar melhor, identificar imóveis anunciados abaixo do valor de mercado e saber onde concentrar a captação.

### Origem da base
A base é **sintética** e gerada pela própria solução (`scripts/generator.py`), para respeitar a LGPD: nenhum dado é de pessoa real. O generator roda na VM `generator`, cria entre 50 e 200 imóveis a cada 30 minutos e publica um CSV no GitHub. Para dar volume às análises, foi feita também uma **carga histórica** única de 20 mil registros (`--historico 20000 --seed 2026`).

O generator **injeta sujeira de propósito** (cerca de 10% dos registros): preço e área negativos ou zerados, quartos impossíveis (-1, 99) e o estado escrito de 6 formas diferentes. Isso permite praticar a limpeza.

In [4]:
partes = []
for nome in arquivos_raw:
    df = pd.read_csv(baixar("data/raw", nome), keep_default_na=True, na_values=[""])
    df["arquivo"] = nome
    partes.append(df)
bruto = pd.concat(partes, ignore_index=True)

print(f"Registros brutos : {len(bruto):,}")
print(f"Arquivos (lotes) : {bruto['arquivo'].nunique()}")
print(f"Período de vendas: {pd.to_datetime(bruto['sale_date']).min():%d/%m/%Y} a {pd.to_datetime(bruto['sale_date']).max():%d/%m/%Y}")
bruto.head()

Registros brutos : 21,393
Arquivos (lotes) : 11
Período de vendas: 01/01/2023 a 28/12/2026


,record_id,seller_id,seller_name,buyer_id,property_type,neighborhood,city,state,latitude,longitude,area_m2,bedrooms,bathrooms,parking_spots,year_built,furnished,condo_fee,price,sale_date,buyer_notes,arquivo
0,df42273b-b21d-4e00-bd56-6eccbf9c4966,S-007,Diego Alves,B-FDE70D23,commercial,Santana,São Paulo,Sao Paulo,-23.50,-46.62,103.20,0.00,1,1,1995,False,832.73,"982,375.53",2024-08-18,Asked about pet policy in the building.,listings_20260925_005450.csv
1,deca2836-5d96-4ea5-a09f-e202f334d5b1,S-005,Marcelo Ferreira,B-094B1EE4,house,Ipiranga,São Paulo,Sao Paulo,-23.60,-46.60,95.40,3.00,1,0,2009,False,0.00,"1,075,329.12",2024-03-08,"Interested in the view, wants to negotiate price.",listings_20260925_005450.csv
2,56c293a0-6b3c-43f5-8a85-dcd38f2805e4,S-016,Aline Gomes,B-3B35413E,penthouse,Itaim Bibi,São Paulo,S.Paulo,-23.59,-46.67,153.10,4.00,4,3,2007,True,"1,783.73","3,442,852.59",2024-03-16,"Interested in the view, wants to negotiate price.",listings_20260925_005450.csv
3,6d22f4ad-171f-48d6-9eec-9f10ba5586bc,S-004,Fernanda Costa,B-8C020AF0,house,Jabaquara,São Paulo,sp,-23.65,-46.64,254.90,5.00,1,1,2011,True,0.00,"2,711,772.53",2024-03-06,Needs parking for two cars.,listings_20260925_005450.csv
4,b890cf24-b57e-43d9-a700-13964ee1459a,S-016,Aline Gomes,B-9F285071,house,Cidade Dutra,São Paulo,SP,-23.70,-46.66,256.10,2.00,1,3,1989,False,0.00,"1,137,479.06",2026-05-25,Looking for a property near public transport.,listings_20260925_005450.csv


### Dicionário de dados

| Variável | Tipo | Descrição | Observações |
|---|---|---|---|
| `record_id` | texto (UUID) | Identificador único do imóvel | chave primária |
| `seller_id` / `seller_name` | texto | Corretor responsável | **nome não entra no DW (LGPD)** |
| `buyer_id` | texto | Comprador | vazio quando o imóvel não foi vendido |
| `property_type` | categórica | apartment, house, studio, commercial, penthouse | |
| `neighborhood` | categórica | Bairro (45 bairros de SP e região) | |
| `city` / `state` | categórica | Cidade e estado | `state` vem com 6 grafias diferentes |
| `latitude` / `longitude` | numérica contínua | Coordenadas do imóvel | |
| `area_m2` | numérica contínua | Área útil em m² | pode vir negativa, zero ou nula (sujeira) |
| `bedrooms` / `bathrooms` | numérica discreta | Quartos e banheiros | quartos podem vir -1, -2 ou 99 (sujeira) |
| `parking_spots` | numérica discreta | Vagas de garagem | |
| `year_built` | numérica discreta | Ano de construção | |
| `furnished` | booleana | Mobiliado | |
| `condo_fee` | numérica contínua | Condomínio (R$) | 0 para casas |
| `price` | numérica contínua | Preço (R$) — **variável alvo** | pode vir negativo, zero ou nulo (sujeira) |
| `sale_date` | data | Data da venda | vazia quando não vendido |
| `buyer_notes` | texto livre | Observação do comprador | dado **semiestruturado** |

### Relação com os 5 Vs do Big Data

In [5]:
lotes = bruto[bruto["arquivo"] != "listings_20261111_000000.csv"].groupby("arquivo").size()
sujos = (
    (bruto["price"].isna() | (bruto["price"] <= 0))
    | (bruto["area_m2"].isna() | (bruto["area_m2"] <= 0))
    | (bruto["bedrooms"].notna() & ((bruto["bedrooms"] < 0) | (bruto["bedrooms"] > 20)))
)
cinco_vs = pd.DataFrame({
    "V": ["Volume", "Velocidade", "Variedade", "Veracidade", "Valor"],
    "Evidência no projeto": [
        f"{len(bruto):,} registros em {bruto['arquivo'].nunique()} lotes; cresce sem parar",
        f"novo lote a cada 30 min (média de {lotes.mean():.0f} imóveis por lote) ≈ {lotes.mean() * 48:,.0f}/dia",
        "dados estruturados (CSV/parquet), texto livre (buyer_notes) e coordenadas geográficas",
        f"{sujos.mean():.1%} dos registros com valores inválidos; estado com {bruto['state'].nunique()} grafias",
        "preço justo por imóvel, oportunidades abaixo do mercado e regiões mais valorizadas",
    ],
})
cinco_vs

,V,Evidência no projeto
0,Volume,"21,393 registros em 11 lotes; cresce sem parar"
1,Velocidade,novo lote a cada 30 min (média de 139 imóveis ...
2,Variedade,"dados estruturados (CSV/parquet), texto livre ..."
3,Veracidade,9.1% dos registros com valores inválidos; esta...
4,Valor,"preço justo por imóvel, oportunidades abaixo d..."


**Limitações da base.** Por ser sintética, o preço segue uma regra do próprio generator (área × preço do m² sorteado dentro de uma faixa por tipo de região). Os padrões encontrados são, portanto, os que o gerador cria; métricas de modelo tendem a ser melhores do que seriam com dados reais. Isso é discutido na seção 6.

<a id="2"></a>
## 2. Análise Exploratória de Dados (EDA)

### 2.1 Tipos de variáveis

In [ ]:
# Classificação de cada variável (a tabela mostra também o tipo que o pandas leu, os nulos e um exemplo).
CLASSE = {
    "record_id": "identificador", "seller_id": "identificador", "buyer_id": "identificador",
    "seller_name": "texto (dado pessoal)", "buyer_notes": "texto livre",
    "property_type": "categórica nominal", "neighborhood": "categórica nominal",
    "city": "categórica nominal", "state": "categórica nominal",
    "latitude": "numérica contínua", "longitude": "numérica contínua", "area_m2": "numérica contínua",
    "condo_fee": "numérica contínua", "price": "numérica contínua (alvo)",
    "bedrooms": "numérica discreta", "bathrooms": "numérica discreta", "parking_spots": "numérica discreta",
    "year_built": "numérica discreta (ano)", "furnished": "booleana", "sale_date": "data",
}
tipos = pd.DataFrame({
    "classificação": pd.Series(CLASSE),
    "tipo pandas": bruto.dtypes.astype(str),
    "valores únicos": bruto.nunique(),
    "nulos": bruto.isna().sum(),
    "exemplo": bruto.iloc[0],
}).drop(index="arquivo", errors="ignore")
tipos

### 2.2 Qualidade dos dados: nulos, duplicidades e valores fora do padrão

In [ ]:
# Funções auxiliares usadas nas próximas seções e dicionário que guarda os indicadores da análise final (seção 6)
def brl(valor):
    """Formata um número como moeda brasileira (R$ 1.234.567)."""
    return "R$ " + f"{valor:,.0f}".replace(",", ".")

RESULTADOS = {}

dados_brutos = bruto.drop(columns="arquivo")

# ---- Nulos por coluna ----
nulos = (dados_brutos.isna().sum().to_frame("nulos").assign(pct=lambda d: d["nulos"] / len(dados_brutos) * 100)
         .query("nulos > 0").sort_values("nulos", ascending=False))
display(nulos.style.background_gradient(cmap="Reds").format({"nulos": "{:,.0f}", "pct": "{:.1f}%"}))

# ---- Duplicidades ----
print("record_id duplicados  :", dados_brutos["record_id"].duplicated().sum())
print("linhas 100% duplicadas:", dados_brutos.duplicated().sum())

# ---- Regras de consistência: quantos registros violam cada regra ----
hoje = pd.Timestamp.today().normalize()
data_venda = pd.to_datetime(dados_brutos["sale_date"], errors="coerce")
regras = pd.DataFrame([
    ("price nulo ou ≤ 0 (alvo inválido)",          (dados_brutos["price"].isna() | (dados_brutos["price"] <= 0)).sum(),         "remover a linha"),
    ("area_m2 nula ou ≤ 0",                         (dados_brutos["area_m2"].isna() | (dados_brutos["area_m2"] <= 0)).sum(),     "remover a linha"),
    ("bedrooms < 0 ou > 20 (sentinelas -1, -2, 99)", ((dados_brutos["bedrooms"] < 0) | (dados_brutos["bedrooms"] > 20)).sum(),   "remover a linha"),
    ("bathrooms < 0 ou > 10",                       ((dados_brutos["bathrooms"] < 0) | (dados_brutos["bathrooms"] > 10)).sum(), "apenas monitorar"),
    ("parking_spots < 0",                           (dados_brutos["parking_spots"] < 0).sum(),                                   "apenas monitorar"),
    ("condo_fee < 0",                               (dados_brutos["condo_fee"] < 0).sum(),                                       "apenas monitorar"),
    ("year_built fora de 1900 até o ano atual",     ((dados_brutos["year_built"] < 1900) | (dados_brutos["year_built"] > hoje.year)).sum(), "apenas monitorar"),
    ("coordenadas fora da Grande São Paulo (aprox.)", ((dados_brutos["latitude"] < -24.1) | (dados_brutos["latitude"] > -23.1) |
                                                       (dados_brutos["longitude"] < -47.1) | (dados_brutos["longitude"] > -46.0)).sum(), "apenas monitorar"),
    ("sale_date no futuro",                         (data_venda > hoje).sum(),                                                   "manter (ver limitações)"),
    ("state: grafias diferentes para o mesmo estado", dados_brutos["state"].nunique(),                                           "padronizar para SP"),
], columns=["Regra", "Registros afetados", "Decisão"])
regras["% da base"] = regras["Registros afetados"] / len(dados_brutos) * 100
regras.loc[regras["Regra"].str.startswith("state"), "% da base"] = np.nan     # nesta linha o número é de grafias, não de registros
display(regras.style.hide(axis="index").format({"% da base": "{:.1f}%"}, na_rep="–")
        .background_gradient(subset=["Registros afetados"], cmap="Oranges"))

### 2.3 Limpeza dos dados (decisões documentadas)
As regras são as **mesmas do pipeline** (limpeza em R na VM `processing`), para que as duas implementações cheguem ao mesmo número de registros.

| Problema encontrado | Decisão |
|---|---|
| `price` nulo ou ≤ 0 (alvo) | remover a linha (não se imputa o alvo) |
| `area_m2` nula ou ≤ 0 | remover a linha (sem área não há preço por m²) |
| `bedrooms` com -1, -2 e 99 | remover a linha; quartos nulos são mantidos (imputados na modelagem) |
| `state` escrito de várias formas (`SP`, `sp`, `S.Paulo`...) | padronizar para `SP` |
| `sale_date` vazia | não é erro: o imóvel ainda não foi vendido; vira a variável `vendido` |
| `sale_date` no futuro | manter: preço e área são válidos; o problema fica registrado como limitação |
| `buyer_id`, `buyer_notes` e `sale_date` só existem **depois** da negociação | não usar para prever o preço (vazamento de informação) |
| `seller_name` | dado pessoal: não usado nas análises (LGPD) |
| `record_id` repetido | manter a primeira ocorrência |

A região de cada bairro (Nobre, Intermediária, Periférica) vem do próprio `generator.py`.

In [ ]:
# Região de cada bairro: lida do generator.py (clone local na VM, GitHub no Colab)
if REPO_LOCAL:
    fonte_generator = open(os.path.join(REPO_LOCAL, "scripts", "generator.py"), encoding="utf-8").read()
else:
    fonte_generator = urllib.request.urlopen(f"{RAW_BASE}/scripts/generator.py", timeout=30).read().decode()
for no in ast.parse(fonte_generator).body:
    if isinstance(no, ast.Assign) and isinstance(no.targets[0], ast.Name) and no.targets[0].id == "SP_NEIGHBORHOODS":
        BAIRROS = ast.literal_eval(no.value)
REGIAO_PT = {"noble": "Nobre", "mid": "Intermediária", "peripheral": "Periférica"}
REGIAO_DO_BAIRRO = {nome: REGIAO_PT[zona] for nome, _, _, zona in BAIRROS}
ordem_regioes = ["Nobre", "Intermediária", "Periférica"]
print(f"{len(BAIRROS)} bairros no generator | regiões:", pd.Series(REGIAO_DO_BAIRRO).value_counts().to_dict())

In [ ]:
def limpar_dados(tabela):
    """Aplica as regras de limpeza documentadas e devolve uma nova tabela (a original não é alterada)."""
    d = tabela.copy()
    d = d.drop_duplicates(subset="record_id", keep="first")

    d["state"] = "SP"                                                          # padroniza o estado
    d["sale_date"] = pd.to_datetime(d["sale_date"], errors="coerce")           # texto -> data

    # remove linhas com valores inválidos (as mesmas regras do R no pipeline)
    d = d[d["price"].notna() & (d["price"] > 0)]
    d = d[d["area_m2"].notna() & (d["area_m2"] > 0)]
    d = d[d["bedrooms"].isna() | d["bedrooms"].between(0, 20)]

    # variáveis derivadas
    d["preco_m2"] = d["price"] / d["area_m2"]
    d["regiao"] = d["neighborhood"].map(REGIAO_DO_BAIRRO)
    d["vendido"] = d["sale_date"].notna()
    return d.reset_index(drop=True)


limpo = limpar_dados(bruto)

linhas_parquet = sum(len(pd.read_parquet(baixar("data/processed", n))) for n in arquivos_processed)
print(f"Antes da limpeza : {len(bruto):,}")
print(f"Depois (pandas)  : {len(limpo):,}  ({1 - len(limpo) / len(bruto):.1%} removidos)")
print(f"Parquets do R    : {linhas_parquet:,}  ->  {'iguais' if linhas_parquet == len(limpo) else 'DIFERENTES: conferir as regras'}")
print("Bairros sem região:", int(limpo["regiao"].isna().sum()))
RESULTADOS["registros_brutos"] = len(bruto)
RESULTADOS["registros_validos"] = len(limpo)

### 2.4 Estatística descritiva

In [ ]:
numericas = ["price", "area_m2", "preco_m2", "bedrooms", "bathrooms", "parking_spots", "year_built", "condo_fee"]
desc = limpo[numericas].describe().T
desc["mediana"] = limpo[numericas].median()
desc["assimetria"] = limpo[numericas].skew()
desc = desc[["count", "mean", "mediana", "std", "min", "25%", "75%", "max", "assimetria"]]
display(desc.style.format("{:,.2f}").background_gradient(subset=["assimetria"], cmap="coolwarm", vmin=-3, vmax=3))

print("\nVariáveis categóricas")
for coluna in ["property_type", "regiao"]:
    tabela = limpo[coluna].value_counts().to_frame("imóveis").assign(pct=lambda t: t["imóveis"] / t["imóveis"].sum() * 100)
    display(tabela.style.format({"imóveis": "{:,.0f}", "pct": "{:.1f}%"}).bar(subset=["imóveis"], color="#5fa8d3"))

# ---- Outliers pela regra do IQR (1,5 × amplitude interquartil) ----
def contar_outliers(serie):
    q1, q3 = serie.quantile([.25, .75])
    limite_inf, limite_sup = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    return limite_inf, limite_sup, int(((serie < limite_inf) | (serie > limite_sup)).sum())

linhas = []
for coluna in ["price", "area_m2", "preco_m2", "condo_fee"]:
    inf, sup, qtd = contar_outliers(limpo[coluna])
    linhas.append((coluna, "base toda", inf, sup, qtd))
# o preço por m² tem patamar diferente em cada região: o IQR faz mais sentido dentro de cada uma
dentro_regiao = sum(contar_outliers(g["preco_m2"])[2] for _, g in limpo.groupby("regiao"))
linhas.append(("preco_m2", "dentro de cada região", np.nan, np.nan, dentro_regiao))
outliers = pd.DataFrame(linhas, columns=["variável", "referência", "limite inferior", "limite superior", "outliers"])
outliers["% da base"] = outliers["outliers"] / len(limpo) * 100
display(outliers.style.hide(axis="index").format({"limite inferior": "{:,.0f}", "limite superior": "{:,.0f}", "% da base": "{:.1f}%"}, na_rep="–"))

### 2.5 Visualizações e interpretação

In [ ]:
# Gráfico 1 – Valores ausentes ANTES da limpeza (qualidade dos dados)
fig, ax = plt.subplots(figsize=(7, 3.4))
miss = (dados_brutos.isna().mean() * 100)[lambda s: s > 0].sort_values()
miss.plot.barh(ax=ax, color="#e07a5f")
ax.set_xlabel("% de valores ausentes"); ax.set_title("Gráfico 1 – Valores ausentes por coluna (base bruta)")
ax.set_xlim(0, max(miss.max() * 1.2, 5))
for i, v in enumerate(miss):
    ax.text(v + .3, i, f"{v:.1f}%", va="center")
plt.tight_layout(); plt.show()

estruturais = [c for c in miss.index if c in ("buyer_id", "sale_date", "buyer_notes")]
falhas = [c for c in miss.index if c not in estruturais]
texto = (f"Interpretação: {', '.join(estruturais)} têm cerca de {miss[estruturais].mean():.0f}% de nulos. São ausências estruturais: "
         "o imóvel ainda não foi vendido, então não há comprador, data nem observação. Isso não é erro e vira a variável 'vendido'; "
         "usar esses campos para prever o preço seria vazamento de informação. ")
if falhas:
    texto += (f"Já {', '.join(falhas)} têm entre {miss[falhas].min():.1f}% e {miss[falhas].max():.1f}% de nulos: são falhas de coleta, "
              "tratadas na limpeza.")
print(texto)

In [ ]:
# Gráfico 2 – Distribuição do preço (escala normal x log)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(limpo["price"] / 1e6, bins=40, kde=True, ax=ax[0], color="#3d405b"); ax[0].set_title("Preço (R$ milhões)"); ax[0].set_xlabel("")
sns.histplot(np.log(limpo["price"]), bins=40, kde=True, ax=ax[1], color="#81b29a"); ax[1].set_title("log(Preço)"); ax[1].set_xlabel("")
fig.suptitle("Gráfico 2 – Distribuição do preço", fontweight="bold"); plt.tight_layout(); plt.show()

print(f"Interpretação: a assimetria do preço é {limpo['price'].skew():.2f} (cauda longa à direita): a mediana é {brl(limpo['price'].median())} "
      f"e a média {brl(limpo['price'].mean())}, puxada por poucos imóveis muito caros. Após o log a assimetria cai para "
      f"{np.log(limpo['price']).skew():.2f}. A média sozinha engana, então usamos a mediana e o preço por m² nas comparações, "
      "e o modelo de ML (seção 4) é treinado com log(preço): o erro passa a ser proporcional (%).")

In [ ]:
# Gráfico 3 – Preço por m² por região e preço por tipo de imóvel (escala log)
ordem_tipo = limpo.groupby("property_type")["price"].median().sort_values().index
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
sns.boxplot(data=limpo, x="regiao", y="preco_m2", order=ordem_regioes, ax=ax[0], palette="deep")
ax[0].set_title("Preço por m² por região"); ax[0].set_xlabel(""); ax[0].set_ylabel("R$/m²")
sns.boxplot(data=limpo, x="property_type", y="price", order=ordem_tipo, ax=ax[1], palette="deep"); ax[1].set_yscale("log")
ax[1].set_title("Preço por tipo (escala log)"); ax[1].set_xlabel(""); ax[1].set_ylabel("R$")
ax[1].tick_params(axis="x", rotation=20)
fig.suptitle("Gráfico 3 – Preço por região e por tipo", fontweight="bold"); plt.tight_layout(); plt.show()

m2_regiao = limpo.groupby("regiao")["preco_m2"].median().reindex(ordem_regioes)
preco_tipo = limpo.groupby("property_type")["price"].median().sort_values()
RESULTADOS["razao_nobre_periferica"] = m2_regiao["Nobre"] / m2_regiao["Periférica"]
print(f"Interpretação: a região é o principal fator do preço por m²: mediana de {brl(m2_regiao['Nobre'])} na Nobre, "
      f"{brl(m2_regiao['Intermediária'])} na Intermediária e {brl(m2_regiao['Periférica'])} na Periférica, ou seja, a Nobre custa "
      f"{RESULTADOS['razao_nobre_periferica']:.1f}× a Periférica. Em preço total, o tipo mais caro é \"{preco_tipo.index[-1]}\" "
      f"(mediana {brl(preco_tipo.iloc[-1])}) e o mais barato é \"{preco_tipo.index[0]}\" ({brl(preco_tipo.iloc[0])}), "
      "diferença que vem principalmente do tamanho típico de cada tipo (gráfico 4).")

In [ ]:
# Gráfico 4 – Quantidade e área por tipo de imóvel
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
limpo["property_type"].value_counts().sort_values().plot.barh(ax=ax[0], color="#3d405b")
ax[0].set_title("Imóveis por tipo"); ax[0].set_xlabel("imóveis")
sns.boxplot(data=limpo, x="property_type", y="area_m2", order=ordem_tipo, ax=ax[1], palette="deep")
ax[1].set_title("Área por tipo"); ax[1].set_xlabel(""); ax[1].set_ylabel("m²"); ax[1].tick_params(axis="x", rotation=20)
fig.suptitle("Gráfico 4 – Tipos de imóvel", fontweight="bold"); plt.tight_layout(); plt.show()

share = limpo["property_type"].value_counts(normalize=True) * 100
area_tipo = limpo.groupby("property_type")["area_m2"].median().sort_values()
print(f"Interpretação: os tipos aparecem em proporções entre {share.min():.0f}% e {share.max():.0f}% (o gerador sorteia o tipo de forma "
      f"quase uniforme; em dados reais, apartamentos dominariam). A área muda muito por tipo: mediana de {area_tipo.iloc[0]:.0f} m² em "
      f"\"{area_tipo.index[0]}\" contra {area_tipo.iloc[-1]:.0f} m² em \"{area_tipo.index[-1]}\". Isso explica por que o preço total varia "
      "tanto entre tipos mesmo com preço por m² parecido.")

In [ ]:
# Gráfico 5 – Relação entre área e preço, por região (amostra de 4.000 imóveis para o gráfico não ficar pesado)
amostra = limpo.sample(min(4000, len(limpo)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=amostra, x="area_m2", y="price", hue="regiao", hue_order=ordem_regioes, alpha=.5, s=18, ax=ax)
ax.set_title("Gráfico 5 – Área x preço por região"); ax.set_xlabel("Área (m²)"); ax.set_ylabel("Preço (R$)")
plt.tight_layout(); plt.show()

corr_area = limpo[["area_m2", "price"]].corr(method="spearman").iloc[0, 1]
print(f"Interpretação: correlação de Spearman entre área e preço = {corr_area:.2f}. O preço cresce com a área, mas em leques separados "
      "por região: o que muda de uma região para outra é a inclinação, ou seja, o preço do m². Isso é uma interação entre área e região: "
      "um modelo linear, que só soma efeitos, tem dificuldade para representá-la; modelos de árvores conseguem (seção 4.2).")

In [ ]:
# Gráfico 6 – Mapa de dispersão (latitude/longitude) colorido pelo preço por m²
fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(amostra["longitude"], amostra["latitude"], c=amostra["preco_m2"], cmap="viridis_r",
                s=amostra["area_m2"].clip(upper=300) / 12 + 4, alpha=.6, edgecolor="none")
plt.colorbar(sc, label="R$/m²"); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Gráfico 6 – Localização dos imóveis (cor = R$/m², tamanho = área)"); plt.tight_layout(); plt.show()

corr_geo = limpo[["latitude", "longitude", "preco_m2"]].corr(method="spearman")["preco_m2"]
print(f"Interpretação: correlação de Spearman do preço/m² com a latitude = {corr_geo['latitude']:.2f} e com a longitude = "
      f"{corr_geo['longitude']:.2f}. A correlação mede só relações monotônicas; a localização pesa por regiões (efeito não linear), "
      "e os modelos de árvores conseguem aproveitar latitude e longitude para isso.")

In [ ]:
# Gráfico 7 – Correlação entre variáveis numéricas
colunas_corr = ["price", "area_m2", "preco_m2", "bedrooms", "bathrooms", "parking_spots", "year_built", "condo_fee", "latitude", "longitude"]
corr = limpo[colunas_corr].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8.5, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Gráfico 7 – Correlação de Spearman"); plt.tight_layout(); plt.show()

corr_preco = corr["price"].drop(["price", "preco_m2"]).abs().sort_values(ascending=False)
fracas = corr_preco[corr_preco < 0.1].index.tolist()
print(f"Interpretação: a variável mais associada ao preço é {corr_preco.index[0]} ({corr['price'][corr_preco.index[0]]:.2f}). "
      + (f"{', '.join(fracas)} têm correlação próxima de zero (menos de 0,10) com o preço. " if fracas else "")
      + "Correlação não é causalidade: o modelo da seção 4.2 mede a contribuição de cada variável considerando todas ao mesmo tempo.")

In [ ]:
# Gráfico 8 – Distribuição das demais variáveis numéricas
colunas_hist = ["area_m2", "bedrooms", "bathrooms", "parking_spots", "year_built", "condo_fee"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, coluna in zip(axes.ravel(), colunas_hist):
    sns.histplot(limpo[coluna].dropna(), bins=25, ax=ax, color="#81b29a"); ax.set_title(coluna); ax.set_xlabel("")
fig.suptitle("Gráfico 8 – Distribuição das variáveis numéricas", fontweight="bold"); plt.tight_layout(); plt.show()

sem_condominio = (limpo["condo_fee"] == 0).mean() * 100
print(f"Interpretação: {sem_condominio:.0f}% dos imóveis têm condomínio zerado (casas e imóveis sem taxa), o que gera o pico em 0 em "
      "condo_fee. Quartos, banheiros, vagas e ano de construção têm distribuição aproximadamente uniforme: são sorteados pelo gerador, "
      "sem concentração nem valores isolados.")

In [ ]:
# Gráfico 9 – Preço por m² por bairro (mediana), colorido pela região
por_bairro = (limpo.groupby(["neighborhood", "regiao"])["preco_m2"].agg(["median", "size"]).reset_index()
                   .rename(columns={"median": "preco_m2_mediano", "size": "imoveis"}).sort_values("preco_m2_mediano"))
cores = dict(zip(ordem_regioes, sns.color_palette("deep", 3)))
fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(por_bairro["neighborhood"], por_bairro["preco_m2_mediano"], color=por_bairro["regiao"].map(cores))
ax.set_xlabel("R$/m² (mediana)"); ax.set_title("Gráfico 9 – Preço por m² por bairro")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=cores[r]) for r in ordem_regioes], labels=ordem_regioes, loc="lower right")
plt.tight_layout(); plt.show()

RESULTADOS["bairro_top_m2"] = f"{por_bairro['neighborhood'].iloc[-1]} ({brl(por_bairro['preco_m2_mediano'].iloc[-1])}/m²)"
RESULTADOS["bairro_base_m2"] = f"{por_bairro['neighborhood'].iloc[0]} ({brl(por_bairro['preco_m2_mediano'].iloc[0])}/m²)"
RESULTADOS["preco_mediano"] = limpo["price"].median()
RESULTADOS["preco_m2_mediano"] = limpo["preco_m2"].median()
print(f"Interpretação: o bairro mais caro por m² é {RESULTADOS['bairro_top_m2']} e o mais barato é {RESULTADOS['bairro_base_m2']}. "
      f"Cada bairro tem em média {por_bairro['imoveis'].mean():,.0f} imóveis na base, o suficiente para uma mediana estável. "
      "As cores mostram que o ranking se organiza em degraus por região: a região explica grande parte da diferença entre bairros.")

<a id="3"></a>
## 3. Armazenamento, ETL e arquitetura analítica

### 3.1 Por que Data Warehouse (e não Data Lake)
Os dados são **predominantemente estruturados**: schema fixo de 20 colunas, tipos bem definidos e perguntas de negócio feitas sempre pelas mesmas dimensões (onde, quando, que tipo de imóvel, quais características). Esse é o cenário de um **Data Warehouse** com modelo dimensional. O único campo semiestruturado (`buyer_notes`) não é usado nas análises; ele e os CSVs brutos ficam preservados no repositório, que funciona como **área de entrada (staging)**: `data/raw` (bruto) → `data/processed` (tratado) → DW.

### 3.2 Arquitetura da solução

```
VM generator ──CSV──▶ GitHub data/raw ──GitHub Actions──▶ VM processing (R: limpeza)
                                                                 │ parquet
                                                                 ▼
                     GitHub data/processed ──git pull──▶ VM bigdata: ETL (Python, a cada 10 min)
                                                                 │
                                                                 ▼
             VM monitoring: Grafana ◀── PostgreSQL (esquema estrela) ──▶ Jupyter / Colab (este notebook)
```

### 3.3 Modelo dimensional (esquema estrela)

| Tabela | Tipo | Conteúdo | Hierarquia |
|---|---|---|---|
| `fato_imovel` | fato | preço, área, preço por m², condomínio, vendido | grão: 1 imóvel |
| `dim_localizacao` | dimensão | bairro, região, cidade, estado | bairro → região → cidade → estado |
| `dim_tempo` | dimensão | data da venda; membro "Sem data" para não vendidos | data → mês → trimestre → ano |
| `dim_tipo_imovel` | dimensão | tipo em inglês e português | — |
| `dim_caracteristicas` | dimensão | quartos, banheiros, vagas, mobiliado, faixa de área e de idade | — |
| `etl_carga` | controle | arquivo, linhas lidas/inseridas/ignoradas, horário | evidência do ETL |

### 3.4 Processo ETL
* **Extração:** o ETL (`scripts/etl.py`) faz `git pull` e lista os parquets que ainda não constam em `etl_carga`.
* **Transformação:** deriva região, preço por m², faixas de área e idade e a dimensão tempo; descarta nome de vendedor e comprador (LGPD).
* **Carga:** insere os membros novos das dimensões e depois a fato, numa transação por arquivo. Cada arquivo entra uma única vez.

No modo `dw` as consultas abaixo vão ao PostgreSQL real. No modo `github` a mesma transformação é refeita aqui e o esquema estrela é montado em DuckDB.

In [18]:
TIPO_PT = {"apartment": "Apartamento", "house": "Casa", "studio": "Studio",
           "commercial": "Comercial", "penthouse": "Cobertura"}
MESES = ["Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho", "Julho",
         "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]


def faixa_area(a):
    return ("até 40 m²" if a <= 40 else "41 a 80 m²" if a <= 80 else "81 a 120 m²" if a <= 120
            else "121 a 200 m²" if a <= 200 else "acima de 200 m²")


def faixa_idade(ano):
    if pd.isna(ano):
        return "Desconhecida"
    idade = pd.Timestamp.today().year - ano
    return ("até 10 anos" if idade <= 10 else "11 a 20 anos" if idade <= 20 else "21 a 30 anos" if idade <= 30
            else "31 a 40 anos" if idade <= 40 else "mais de 40 anos")


def montar_dw_em_memoria():
    """Réplica do ETL (scripts/etl.py) em pandas + DuckDB, para rodar sem a VM."""
    partes = []
    for i, nome in enumerate(arquivos_processed, start=1):
        p = pd.read_parquet(baixar("data/processed", nome))
        p["id_carga"] = i
        partes.append(p)
    t = pd.concat(partes, ignore_index=True).drop_duplicates("record_id")
    t["sale_date"] = pd.to_datetime(t["sale_date"], errors="coerce")

    loc = pd.DataFrame([(n, REGIAO_PT[z], "São Paulo", "SP", la, lo) for n, la, lo, z in BAIRROS],
                       columns=["bairro", "regiao", "cidade", "estado", "latitude", "longitude"])
    loc.insert(0, "sk_localizacao", range(1, len(loc) + 1))

    tipo = pd.DataFrame(sorted(TIPO_PT.items()), columns=["tipo", "tipo_pt"])
    tipo.insert(0, "sk_tipo", range(1, len(tipo) + 1))

    datas = t["sale_date"].dropna().dt.normalize().drop_duplicates()
    tempo = pd.DataFrame({
        "sk_tempo": datas.dt.strftime("%Y%m%d").astype(int), "data": datas.dt.date, "ano": datas.dt.year,
        "trimestre": datas.dt.quarter, "mes": datas.dt.month, "nome_mes": datas.dt.month.map(lambda m: MESES[m - 1]),
    })
    tempo = pd.concat([pd.DataFrame([{"sk_tempo": 0, "nome_mes": "Sem data"}]), tempo], ignore_index=True)

    t["faixa_area"] = t["area_m2"].map(faixa_area)
    t["faixa_idade"] = t["year_built"].map(faixa_idade)
    chave_carac = ["bedrooms", "bathrooms", "parking_spots", "furnished", "faixa_area", "faixa_idade"]
    carac = t[chave_carac].drop_duplicates().reset_index(drop=True)
    carac.insert(0, "sk_caracteristicas", range(1, len(carac) + 1))

    fato = (t.merge(loc[["sk_localizacao", "bairro"]], left_on="neighborhood", right_on="bairro")
             .merge(tipo[["sk_tipo", "tipo"]], left_on="property_type", right_on="tipo")
             .merge(carac, on=chave_carac, how="left"))
    fato["sk_tempo"] = fato["sale_date"].dt.strftime("%Y%m%d").fillna("0").astype(int)
    fato = pd.DataFrame({
        "record_id": fato["record_id"], "sk_localizacao": fato["sk_localizacao"], "sk_tempo": fato["sk_tempo"],
        "sk_tipo": fato["sk_tipo"], "sk_caracteristicas": fato["sk_caracteristicas"], "id_carga": fato["id_carga"],
        "vendido": fato["sale_date"].notna(), "preco": fato["price"], "area_m2": fato["area_m2"],
        "preco_m2": (fato["price"] / fato["area_m2"]).round(2), "condominio": fato["condo_fee"],
        "ano_construcao": fato["year_built"], "latitude": fato["latitude"], "longitude": fato["longitude"],
    })
    carac = carac.rename(columns={"bedrooms": "quartos", "bathrooms": "banheiros",
                                  "parking_spots": "vagas", "furnished": "mobiliado"})

    con = duckdb.connect()
    con.execute("CREATE SCHEMA dw")
    for nome, df in {"dim_localizacao": loc, "dim_tipo_imovel": tipo, "dim_tempo": tempo,
                     "dim_caracteristicas": carac, "fato_imovel": fato}.items():
        con.register("tmp", df)
        con.execute(f"CREATE TABLE dw.{nome} AS SELECT * FROM tmp")
        con.unregister("tmp")
    con.execute("""
        CREATE VIEW dw.vw_imoveis AS
        SELECT f.record_id, l.bairro, l.regiao, l.cidade, l.estado, t.tipo, t.tipo_pt,
               c.quartos, c.banheiros, c.vagas, c.mobiliado, c.faixa_area, c.faixa_idade,
               tp.data AS data_venda, tp.ano, tp.trimestre, tp.mes, tp.nome_mes,
               f.vendido, f.preco, f.area_m2, f.preco_m2, f.condominio, f.ano_construcao,
               f.latitude, f.longitude, f.id_carga
        FROM dw.fato_imovel f
        JOIN dw.dim_localizacao l USING (sk_localizacao)
        JOIN dw.dim_tipo_imovel t USING (sk_tipo)
        JOIN dw.dim_caracteristicas c USING (sk_caracteristicas)
        JOIN dw.dim_tempo tp USING (sk_tempo)
    """)
    return con


if MODO == "dw":
    from sqlalchemy import create_engine
    engine = create_engine("postgresql+psycopg:///" + os.getenv("DW_DSN", "dbname=realestate").split("dbname=")[-1])
else:
    con = montar_dw_em_memoria()


def consulta(sql):
    """Executa SQL no Data Warehouse ativo (PostgreSQL no modo dw, DuckDB no modo github)."""
    df = pd.read_sql(sql, engine) if MODO == "dw" else con.sql(sql).df()
    for col in df.columns:  # NUMERIC do PostgreSQL chega como Decimal
        if df[col].dtype == object and df[col].map(lambda v: isinstance(v, Decimal)).any():
            df[col] = df[col].astype(float)
    return df


consulta("""
    SELECT 'fato_imovel' AS tabela, COUNT(*) AS linhas FROM dw.fato_imovel
    UNION ALL SELECT 'dim_localizacao', COUNT(*) FROM dw.dim_localizacao
    UNION ALL SELECT 'dim_tempo', COUNT(*) FROM dw.dim_tempo
    UNION ALL SELECT 'dim_tipo_imovel', COUNT(*) FROM dw.dim_tipo_imovel
    UNION ALL SELECT 'dim_caracteristicas', COUNT(*) FROM dw.dim_caracteristicas
""")

,tabela,linhas
0,fato_imovel,19441
1,dim_localizacao,45
2,dim_tempo,1345
3,dim_tipo_imovel,5
4,dim_caracteristicas,2714


In [19]:
if MODO == "dw":
    display(consulta("""SELECT id, arquivo, finalizado_em, linhas_lidas, linhas_inseridas, linhas_ignoradas, status
                         FROM dw.etl_carga ORDER BY id DESC LIMIT 10"""))
else:
    print("Modo github: o histórico de cargas (dw.etl_carga) só existe no Data Warehouse real da VM bigdata.")

,id,arquivo,finalizado_em,linhas_lidas,linhas_inseridas,linhas_ignoradas,status
0,11,listings_20261112_000743_clean.parquet,2026-11-12 00:15:22.629662+00:00,137,137,0,concluída
1,10,listings_20261111_233849_clean.parquet,2026-11-11 23:39:37.910363+00:00,157,157,0,concluída
2,9,listings_20261111_230844_clean.parquet,2026-11-11 23:10:25.768528+00:00,148,148,0,concluída
3,8,listings_20261111_000000_clean.parquet,2026-11-11 23:10:23.940992+00:00,18174,18174,0,concluída
4,7,listings_20261111_011624_clean.parquet,2026-11-11 01:17:34.147810+00:00,159,159,0,concluída
5,6,listings_20261111_004622_clean.parquet,2026-11-11 01:17:34.046617+00:00,143,143,0,concluída
6,5,listings_20261111_001618_clean.parquet,2026-11-11 01:17:33.931490+00:00,165,165,0,concluída
7,4,listings_20261110_171513_clean.parquet,2026-11-11 01:17:33.874495+00:00,62,62,0,concluída
8,3,listings_20261110_165046_clean.parquet,2026-11-11 01:17:33.829074+00:00,51,51,0,concluída
9,2,listings_20261006_231642_clean.parquet,2026-11-11 01:17:33.736134+00:00,127,127,0,concluída


### 3.5 Operações OLAP
Todas as consultas usam a view `dw.vw_imoveis`, que junta a fato com as dimensões (o cubo "achatado").

**Roll-up**: sobe na hierarquia de localização, de bairro para região e para o total geral.

In [20]:
rollup = consulta("""
    SELECT COALESCE(regiao, 'TOTAL GERAL') AS regiao,
           COALESCE(bairro, CASE WHEN regiao IS NULL THEN '' ELSE '(total da região)' END) AS bairro,
           COUNT(*) AS imoveis,
           ROUND(AVG(preco_m2), 0) AS preco_m2_medio
    FROM dw.vw_imoveis
    GROUP BY ROLLUP (regiao, bairro)
    ORDER BY GROUPING(regiao), regiao, GROUPING(bairro) DESC, preco_m2_medio DESC
""")
rollup[(rollup["bairro"].isin(["", "(total da região)"]))]

,regiao,bairro,imoveis,preco_m2_medio
0,Intermediária,(total da região),9581,"8,978.00"
23,Nobre,(total da região),6039,"18,509.00"
38,Periférica,(total da região),3821,"4,501.00"
48,TOTAL GERAL,,19441,"11,059.00"


**Drill-down**: desce na hierarquia de tempo, de ano para trimestre e de trimestre para mês (só imóveis vendidos).

In [21]:
por_ano = consulta("""
    SELECT ano, COUNT(*) AS vendas, ROUND(SUM(preco) / 1e6, 1) AS valor_milhoes, ROUND(AVG(preco_m2), 0) AS preco_m2_medio
    FROM dw.vw_imoveis WHERE vendido GROUP BY ano ORDER BY ano
""")
ultimo_ano = int(por_ano["ano"].max())
por_trimestre_dw = consulta(f"""
    SELECT ano, trimestre, COUNT(*) AS vendas, ROUND(AVG(preco_m2), 0) AS preco_m2_medio
    FROM dw.vw_imoveis WHERE vendido AND ano = {ultimo_ano} GROUP BY ano, trimestre ORDER BY trimestre
""")
por_mes = consulta(f"""
    SELECT ano, trimestre, mes, nome_mes, COUNT(*) AS vendas
    FROM dw.vw_imoveis WHERE vendido AND ano = {ultimo_ano} AND trimestre = 1
    GROUP BY ano, trimestre, mes, nome_mes ORDER BY mes
""")
display(por_ano, por_trimestre_dw, por_mes)

,ano,vendas,valor_milhoes,preco_m2_medio
0,2023,3295,"6,078.50","11,095.00"
1,2024,3371,"6,065.30","10,983.00"
2,2025,3520,"6,396.80","11,121.00"
3,2026,3335,"6,197.70","11,054.00"


,ano,trimestre,vendas,preco_m2_medio
0,2026,1,834,"10,995.00"
1,2026,2,815,"10,975.00"
2,2026,3,831,"10,913.00"
3,2026,4,855,"11,322.00"


,ano,trimestre,mes,nome_mes,vendas
0,2026,1,1,Janeiro,279
1,2026,1,2,Fevereiro,292
2,2026,1,3,Março,263


**Slice**: fixa uma dimensão (somente apartamentos) e analisa as demais.

In [22]:
consulta("""
    SELECT regiao, COUNT(*) AS apartamentos, ROUND(AVG(preco), 0) AS preco_medio, ROUND(AVG(preco_m2), 0) AS preco_m2_medio
    FROM dw.vw_imoveis WHERE tipo_pt = 'Apartamento'
    GROUP BY regiao ORDER BY preco_m2_medio DESC
""")

,regiao,apartamentos,preco_medio,preco_m2_medio
0,Nobre,1263,"2,266,113.00","18,625.00"
1,Intermediária,1947,"1,089,836.00","8,933.00"
2,Periférica,807,"545,844.00","4,482.00"


**Dice**: recorta várias dimensões ao mesmo tempo (região Nobre, 2 a 3 quartos, vendidos no último ano).

In [23]:
consulta(f"""
    SELECT bairro, COUNT(*) AS vendas, ROUND(AVG(preco), 0) AS preco_medio, ROUND(AVG(preco_m2), 0) AS preco_m2_medio
    FROM dw.vw_imoveis
    WHERE regiao = 'Nobre' AND quartos BETWEEN 2 AND 3 AND vendido AND ano = {ultimo_ano}
    GROUP BY bairro ORDER BY preco_m2_medio DESC
""")

,bairro,vendas,preco_medio,preco_m2_medio
0,Jardins,20,"3,866,814.00","19,674.00"
1,Perdizes,20,"4,579,528.00","19,397.00"
2,Pinheiros,26,"3,534,162.00","19,282.00"
3,Paraíso,18,"3,857,532.00","19,026.00"
4,Higienópolis,22,"3,603,703.00","18,807.00"
5,Alto de Pinheiros,19,"3,195,328.00","18,684.00"
6,Campo Belo,21,"3,064,859.00","18,626.00"
7,Itaim Bibi,13,"3,617,808.00","18,350.00"
8,Moema,20,"3,039,113.00","18,338.00"
9,Consolação,18,"3,952,765.00","18,245.00"


**Pivot / CUBE**: todas as combinações de tipo × região, com subtotais, giradas em tabela.

In [24]:
cubo = consulta("""
    SELECT COALESCE(tipo_pt, 'TODOS') AS tipo, COALESCE(regiao, 'TODAS') AS regiao,
           ROUND(AVG(preco_m2), 0) AS preco_m2_medio
    FROM dw.vw_imoveis
    GROUP BY CUBE (tipo_pt, regiao)
""")
cubo.pivot(index="tipo", columns="regiao", values="preco_m2_medio")[ordem_regioes + ["TODAS"]]

regiao,Nobre,Intermediária,Periférica,TODAS
tipo,,,,
Apartamento,"18,625.00","8,933.00","4,482.00","11,086.00"
Casa,"18,545.00","9,017.00","4,513.00","11,223.00"
Cobertura,"18,445.00","8,969.00","4,510.00","11,046.00"
Comercial,"18,402.00","9,003.00","4,467.00","11,093.00"
Studio,"18,525.00","8,968.00","4,532.00","10,848.00"
TODOS,"18,509.00","8,978.00","4,501.00","11,059.00"


**Interpretação das operações OLAP.**
* **Roll-up:** o m² da região Nobre custa cerca do dobro da Intermediária e cerca de 4 vezes o da Periférica; dentro de cada região os bairros são parecidos entre si.
* **Drill-down:** o volume de vendas e o preço por m² se mantêm estáveis de ano para ano, de trimestre para trimestre e de mês para mês.
* **Slice (apartamentos):** a mesma hierarquia de regiões se repete quando olhamos só um tipo de imóvel.
* **Dice:** nos bairros nobres, imóveis de 2 a 3 quartos vendidos no último ano têm preço por m² entre cerca de R$ 17 e 20 mil.
* **Cubo:** dentro de uma mesma região, todos os tipos têm preço por m² muito parecido. O tipo de imóvel muda o **preço total** (pela área típica), mas **não o preço do m²**.

**Para a imobiliária:** a precificação deve partir da **localização** (preço do m² da região e do bairro) multiplicada pela **área**; o tipo de imóvel entra pela área, e não como um prêmio no m².

<a id="4"></a>
## 4. Data Mining e Machine Learning

A partir daqui os dados vêm do Data Warehouse (`dw.vw_imoveis`). Usamos duas técnicas, cada uma ligada a uma pergunta do projeto:

| Técnica | Tarefa | Algoritmo | Pergunta que responde |
|---|---|---|---|
| **Data Mining** | Agrupamento (não supervisionado) | **K-Means** | Existem perfis naturais de imóveis na carteira? |
| **Machine Learning** | **Regressão** (supervisionado; alvo `preco` contínuo) | **Random Forest Regressor** | Dá para estimar o preço a partir das características e da localização? |

In [ ]:
imoveis = consulta("SELECT * FROM dw.vw_imoveis")
print(f"{len(imoveis):,} imóveis no DW")

def para_binario(serie):
    """True/False vira 1/0, venha como booleano ou como texto ('True'/'False'), conforme o banco; nulo vira 0."""
    return serie.astype(str).str.strip().str.lower().isin(["true", "t", "1", "1.0", "sim"]).astype(int)

imoveis["vendido"] = para_binario(imoveis["vendido"]).astype(bool)     # usado nas seções 4.5 e 6

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error, mean_squared_error,
                             r2_score, silhouette_score, normalized_mutual_info_score)
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### 4.1 Data Mining: segmentação dos imóveis com K-Means
**Por que K-Means:** é simples, rápido (escala bem) e interpretável: forma grupos de imóveis parecidos pela distância entre eles.

**Variáveis usadas:** **área** e **preço por m²**, as duas que a EDA mostrou ligadas ao valor (gráficos 5 e 7). Ambas entram em **log** (distribuição assimétrica) e **padronizadas** (mesmo peso). Quartos, banheiros, vagas e ano de construção ficam de fora: na EDA (gráfico 8) elas são sorteadas de forma uniforme e sem relação com o preço, então só adicionariam ruído à distância entre imóveis (testamos incluí-las e a silhueta cai, com grupos que não se relacionam com nenhuma região).

**Número de grupos:** o **k** é escolhido pelo **coeficiente de silhueta** (quanto maior, mais bem separados os grupos), entre 3 e 6. Como a base é grande, a silhueta é calculada em uma amostra de 5.000 imóveis.

> A **região não entra** como variável: depois de formar os grupos, medimos o quanto eles coincidem com Nobre / Intermediária / Periférica (NMI: 0 = nenhuma relação, 1 = idênticos). O agrupamento é exploratório e **não** alimenta o modelo de previsão da seção 4.2.

In [ ]:
# ---------- Preparação ----------
dm = imoveis.dropna(subset=["area_m2", "preco_m2"]).copy()
dm["log_area"] = np.log(dm["area_m2"])
dm["log_preco_m2"] = np.log(dm["preco_m2"])
feat_cluster = ["log_area", "log_preco_m2"]                                  # tamanho e valor do m²

Xc = StandardScaler().fit_transform(dm[feat_cluster])                       # mesma escala para as duas variáveis

# ---------- Escolha de k ----------
linhas = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xc)
    linhas.append((k, km.inertia_, silhouette_score(Xc, km.labels_, sample_size=5000, random_state=42)))
tab_k = pd.DataFrame(linhas, columns=["k", "inércia", "silhueta"])
k_final = int(tab_k[tab_k["k"] >= 3].sort_values("silhueta", ascending=False).iloc[0]["k"])
display(tab_k.style.hide(axis="index").format({"inércia": "{:,.0f}", "silhueta": "{:.3f}"})
        .highlight_max(subset=["silhueta"], color="#c7f0c7"))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(tab_k["k"], tab_k["inércia"], "o-"); ax[0].set_title("Método do cotovelo (inércia)"); ax[0].set_xlabel("k")
ax[1].plot(tab_k["k"], tab_k["silhueta"], "o-", color="#e07a5f"); ax[1].set_title("Silhueta"); ax[1].set_xlabel("k")
ax[1].axvline(k_final, ls="--", color="gray"); plt.tight_layout(); plt.show()

RESULTADOS["k_clusters"] = k_final
RESULTADOS["silhueta"] = float(tab_k.loc[tab_k["k"] == k_final, "silhueta"].iloc[0])
qualidade = "fraca" if RESULTADOS["silhueta"] < 0.25 else "razoável" if RESULTADOS["silhueta"] < 0.5 else "forte"
print(f"k escolhido = {k_final} (maior silhueta entre 3 e 6: {RESULTADOS['silhueta']:.3f}).")
print(f"Leitura da silhueta: estrutura de grupos {qualidade} (< 0,25 fraca; 0,25–0,50 razoável; > 0,50 forte)."
      + (" Os grupos se sobrepõem: leia os segmentos como tendências, não como fronteiras rígidas." if qualidade == "fraca" else ""))

In [ ]:
# ---------- Modelo final e perfil dos grupos ----------
km = KMeans(n_clusters=k_final, n_init=10, random_state=42).fit(Xc)
dm["segmento"] = km.labels_

perfil = dm.groupby("segmento").agg(imoveis=("preco", "size"), area_mediana=("area_m2", "median"),
                                    preco_mediano=("preco", "median"), preco_m2_mediano=("preco_m2", "median"))
perfil["tipo_predominante"] = dm.groupby("segmento")["tipo_pt"].agg(lambda s: s.value_counts().idxmax())
perfil["região_predominante"] = dm.groupby("segmento")["regiao"].agg(lambda s: s.value_counts().idxmax())
perfil["% dessa região"] = dm.groupby("segmento")["regiao"].agg(lambda s: s.value_counts(normalize=True).max() * 100)
perfil = perfil.sort_values("preco_mediano")
display(perfil.style.format({"imoveis": "{:,.0f}", "area_mediana": "{:,.0f}", "preco_mediano": "R$ {:,.0f}",
                             "preco_m2_mediano": "R$ {:,.0f}", "% dessa região": "{:.0f}%"})
        .background_gradient(subset=["preco_mediano", "area_mediana"], cmap="YlOrRd").bar(subset=["imoveis"], color="#5fa8d3"))

# ---------- Visualização (PCA em 2 dimensões, só para desenhar) ----------
pca = PCA(n_components=2, random_state=42).fit(Xc)
proj = pca.transform(Xc)
ponto = np.random.RandomState(42).choice(len(dm), size=min(4000, len(dm)), replace=False)
cruzamento = pd.crosstab(dm["segmento"], dm["regiao"], normalize="index").reindex(columns=ordem_regioes).loc[perfil.index] * 100

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
sns.scatterplot(x=proj[ponto, 0], y=proj[ponto, 1], hue=dm["segmento"].iloc[ponto].astype(str), palette="deep", ax=ax[0], s=14, alpha=.6)
ax[0].set_title(f"Segmentos (PCA: {pca.explained_variance_ratio_.sum() * 100:.0f}% da variância)")
ax[0].set_xlabel("Componente 1"); ax[0].set_ylabel("Componente 2"); ax[0].legend(title="segmento", fontsize=8)
sns.boxplot(data=dm, x="segmento", y="preco", order=perfil.index, palette="deep", ax=ax[1]); ax[1].set_yscale("log")
ax[1].set_title("Preço por segmento (escala log)"); ax[1].set_ylabel("R$")
sns.heatmap(cruzamento, annot=True, fmt=".0f", cmap="Blues", ax=ax[2], cbar_kws={"label": "% do segmento"})
ax[2].set_title("Região dentro de cada segmento (%)"); ax[2].set_xlabel(""); ax[2].set_ylabel("segmento")
plt.tight_layout(); plt.show()

nmi = normalized_mutual_info_score(dm["regiao"], dm["segmento"])
RESULTADOS["nmi_regiao"] = nmi
print("Interpretação dos segmentos (do mais barato ao mais caro):")
for seg, r in perfil.iterrows():
    print(f" • Segmento {seg}: {r['imoveis']:,.0f} imóveis, predomínio de \"{r['tipo_predominante']}\" e da região {r['região_predominante']} "
          f"({r['% dessa região']:.0f}%), área mediana {r['area_mediana']:.0f} m², preço mediano {brl(r['preco_mediano'])} "
          f"({brl(r['preco_m2_mediano'])}/m²).")
print(f"Concordância dos segmentos com a região (NMI): {nmi:.2f}. "
      + ("O agrupamento, sem receber a região, reencontrou em boa parte a divisão Nobre / Intermediária / Periférica; o restante vem do tamanho do imóvel."
         if nmi >= 0.4 else
         "Os segmentos refletem mais o tamanho e o nível de preço do m² do que a região em si."))
print("Decisão: cada segmento pode ter política própria de precificação e de divulgação (por exemplo, metas de preço/m² por segmento).")

### 4.2 Machine Learning: regressão do preço com Random Forest
**Por que Random Forest Regressor:**
- captura relações **não lineares** e interações (a área pesa diferente em cada região, como visto no gráfico 5);
- é **robusto a outliers** e funciona bem com variáveis mistas (numéricas + categóricas);
- mede a **importância das variáveis**, o que gera insights de negócio.

**Preparação (decisões):**
- alvo: `preco`, treinado em **log(preço)** (o erro fica proporcional) e avaliado de volta em **R$**;
- entram: área, quartos, banheiros, vagas, condomínio, ano de construção, mobiliado, latitude, longitude, tipo e região;
- ficam de fora: `preco_m2` (é calculado a partir do preço: vazamento do alvo), a data de venda e o comprador (só existem depois da negociação), identificadores e dados pessoais, e `bairro` (a região e as coordenadas já carregam a localização);
- valores ausentes: imputação pela mediana (numéricas) e categoria `missing` (texto), dentro do *pipeline*, para não vazar informação do teste;
- Random Forest com 100 árvores, mínimo de 5 imóveis por folha e metade das variáveis por divisão (`max_features=0.5`), configuração que o grupo escolheu comparando tempo e erro;
- divisão **80% treino / 20% teste** (`random_state=42`) e **validação cruzada de 5 partes** para conferir a estabilidade.

In [ ]:
# ---------- Features e alvo ----------
NUMERICAS = ["latitude", "longitude", "area_m2", "quartos", "banheiros", "vagas", "ano_construcao", "condominio", "mobiliado"]
CATEGORICAS = ["tipo_pt", "regiao"]

base_ml = imoveis.dropna(subset=["preco", "area_m2", "tipo_pt", "regiao"]).copy()
base_ml["mobiliado"] = para_binario(base_ml["mobiliado"])                  # True/False (ou texto) -> 1/0
X = base_ml[NUMERICAS + CATEGORICAS]
y = base_ml["preco"]

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")),
                                      ("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                                          ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, NUMERICAS),
                                               ("cat", categorical_transformer, CATEGORICAS)])

def montar(modelo):
    """Pipeline completa: pré-processamento + modelo, treinando em log(preço) e devolvendo R$."""
    return TransformedTargetRegressor(
        regressor=Pipeline(steps=[("preprocessor", preprocessor), ("regressor", modelo)]),
        func=np.log, inverse_func=np.exp)

def random_forest():
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=5, max_features=0.5, n_jobs=-1, random_state=42)

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Treino: {len(X_treino):,} | Teste: {len(X_teste):,}")

modelos = {"Regressão Linear": montar(LinearRegression()), "Random Forest": montar(random_forest())}
resultados, previsoes = [], {}
for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    previsoes[nome] = modelo.predict(X_teste)
    resultados.append({"modelo": nome,
                       "MAE (R$)": mean_absolute_error(y_teste, previsoes[nome]),
                       "RMSE (R$)": mean_squared_error(y_teste, previsoes[nome]) ** 0.5,
                       "MAPE": mean_absolute_percentage_error(y_teste, previsoes[nome]),
                       "R²": r2_score(y_teste, previsoes[nome])})
metricas = pd.DataFrame(resultados).set_index("modelo")
display(metricas.style.format({"MAE (R$)": "R$ {:,.0f}", "RMSE (R$)": "R$ {:,.0f}", "MAPE": "{:.1%}", "R²": "{:.3f}"})
        .highlight_max(subset=["R²"], color="#c7f0c7").highlight_min(subset=["MAE (R$)", "RMSE (R$)", "MAPE"], color="#c7f0c7"))

pipeline = modelos["Random Forest"]
y_pred = previsoes["Random Forest"]
RESULTADOS.update(r2_teste=metricas.loc["Random Forest", "R²"], mae_teste=metricas.loc["Random Forest", "MAE (R$)"],
                  mape_teste=metricas.loc["Random Forest", "MAPE"] * 100)

# ---------- Gráficos: real x previsto e erro percentual ----------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ponto = np.random.RandomState(0).choice(len(y_teste), size=min(3000, len(y_teste)), replace=False)
lim = [0, max(y_teste.max(), y_pred.max()) / 1e6 * 1.02]
ax[0].scatter(y_teste.iloc[ponto] / 1e6, y_pred[ponto] / 1e6, alpha=.4, s=10); ax[0].plot(lim, lim, "r--")
ax[0].set_xlim(lim); ax[0].set_ylim(lim); ax[0].set_xlabel("Preço real (R$ mi)"); ax[0].set_ylabel("Preço previsto (R$ mi)")
ax[0].set_title("Random Forest: real x previsto (teste)")
erro_pct = ((y_pred / y_teste - 1) * 100).clip(-100, 200)
sns.histplot(erro_pct, bins=40, ax=ax[1], color="#81b29a"); ax[1].axvline(0, color="r", ls="--")
ax[1].set_title("Erro percentual por imóvel (teste)"); ax[1].set_xlabel("(previsto / real − 1) em %, limitado a −100% e +200%")
plt.tight_layout(); plt.show()

rf_m, lin_m = metricas.loc["Random Forest"], metricas.loc["Regressão Linear"]
print(f"Interpretação: R² = {rf_m['R²']:.2f} significa que o Random Forest explica cerca de {rf_m['R²'] * 100:.0f}% da variação dos preços "
      f"no teste ({len(y_teste):,} imóveis), com erro médio de {brl(rf_m['MAE (R$)'])} (MAPE {rf_m['MAPE']:.0%}). A Regressão Linear, "
      f"treinada com o mesmo alvo em log, chega a R² {lin_m['R²']:.2f} e MAPE {lin_m['MAPE']:.0%}.")

#### Validação cruzada e comparação com modelos de referência
Uma única divisão treino/teste pode ser sorte ou azar. Repetimos em 5 partes (*5-fold*) e comparamos com dois modelos de referência: um que **sempre prevê a mediana** (piso) e uma **Regressão Linear**. O Random Forest só se justifica se for melhor que os dois.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
candidatos = {"Mediana (piso)": DummyRegressor(strategy="median"),
              "Regressão Linear": LinearRegression(),
              "Random Forest": random_forest()}
linhas = []
for nome, modelo in candidatos.items():
    r2 = cross_val_score(montar(modelo), X, y, cv=cv, scoring="r2")
    mae = -cross_val_score(montar(modelo), X, y, cv=cv, scoring="neg_mean_absolute_error")
    linhas.append((nome, r2.mean(), r2.std(), mae.mean()))
comparacao = pd.DataFrame(linhas, columns=["Modelo", "R² médio (5-fold)", "Desvio do R²", "MAE médio (R$)"]).set_index("Modelo")
display(comparacao.style.format({"R² médio (5-fold)": "{:.3f}", "Desvio do R²": "{:.3f}", "MAE médio (R$)": "R$ {:,.0f}"})
        .highlight_max(subset=["R² médio (5-fold)"], color="#c7f0c7").highlight_min(subset=["MAE médio (R$)"], color="#c7f0c7"))
RESULTADOS["r2_cv_rf"] = comparacao.loc["Random Forest", "R² médio (5-fold)"]
RESULTADOS["r2_cv_rf_std"] = comparacao.loc["Random Forest", "Desvio do R²"]
RESULTADOS["r2_cv_linear"] = comparacao.loc["Regressão Linear", "R² médio (5-fold)"]

comparacao["R² médio (5-fold)"].plot.barh(xerr=comparacao["Desvio do R²"], figsize=(6.5, 2.8), color=["#bbb", "#e07a5f", "#3d405b"])
plt.xlabel("R² médio (barra de erro = desvio)"); plt.title("Comparação de modelos (validação cruzada)"); plt.tight_layout(); plt.show()
melhor = comparacao["R² médio (5-fold)"].idxmax()
print(f"Interpretação: o melhor modelo na validação cruzada é \"{melhor}\" (R² médio {comparacao.loc[melhor, 'R² médio (5-fold)']:.2f}). "
      f"O Random Forest tem R² {RESULTADOS['r2_cv_rf']:.2f} (desvio {RESULTADOS['r2_cv_rf_std']:.2f}) contra {RESULTADOS['r2_cv_linear']:.2f} "
      f"da Regressão Linear e {comparacao.loc['Mediana (piso)', 'R² médio (5-fold)']:.2f} do modelo que sempre prevê a mediana.")

### 4.3 O que mais pesa no preço? (importância das variáveis)

In [ ]:
rf = pipeline.regressor_.named_steps["regressor"]
nomes = pipeline.regressor_.named_steps["preprocessor"].get_feature_names_out()

def variavel_original(nome):
    """Agrupa as colunas dummy (one-hot) de volta na variável original."""
    resto = nome.split("__", 1)[1]
    if nome.startswith("num__"):
        return resto
    return next((c for c in CATEGORICAS if resto.startswith(c)), resto)

importancias = pd.Series(rf.feature_importances_, index=[variavel_original(n) for n in nomes]).groupby(level=0).sum().sort_values()
ax = importancias.plot.barh(figsize=(7, 4), color="#3d405b"); ax.set_xlabel("Importância relativa")
plt.title("Importância das variáveis (Random Forest)"); plt.tight_layout(); plt.show()

imp_tab = importancias.sort_values(ascending=False).to_frame("importância")
imp_tab["acumulada"] = imp_tab["importância"].cumsum()
display(imp_tab.style.format("{:.1%}").bar(subset=["importância"], color="#5fa8d3"))
RESULTADOS["var_top"] = imp_tab.index[0]
RESULTADOS["var_top_imp"] = imp_tab["importância"].iloc[0]
RESULTADOS["imp_area_regiao"] = imp_tab.loc[[v for v in ("area_m2", "regiao") if v in imp_tab.index], "importância"].sum()
RESULTADOS["imp_tipo"] = imp_tab["importância"].get("tipo_pt", 0.0)
print(f"Interpretação: \"{imp_tab.index[0]}\" é a variável mais importante ({imp_tab['importância'].iloc[0]:.0%}), seguida de "
      f"\"{imp_tab.index[1]}\" ({imp_tab['importância'].iloc[1]:.0%}) e \"{imp_tab.index[2]}\" ({imp_tab['importância'].iloc[2]:.0%}). "
      f"Área e região juntas somam {RESULTADOS['imp_area_regiao']:.0%} da importância; o tipo de imóvel pesa {RESULTADOS['imp_tipo']:.0%}, em boa parte porque "
      "o tipo já carrega a área típica (studio é pequeno, cobertura é grande). Quartos, vagas, idade, condomínio e mobília dividem o restante, "
      "o que confirma as correlações próximas de zero do gráfico 7.")

### 4.4 Simulação: previsão para novos imóveis
Como na aula: criamos imóveis hipotéticos (latitude e longitude do centro da região; as demais características recebem a mediana da base) e pedimos o preço estimado. É o uso prático do modelo para **precificar novos anúncios**. Repare no mesmo apartamento de 120 m² nas três regiões.

In [ ]:
centro = base_ml.groupby("regiao")[["latitude", "longitude"]].median()
descricao = [
    ("Apartamento 120 m² · região Nobre",         dict(tipo_pt="Apartamento", regiao="Nobre",         area_m2=120, quartos=3, banheiros=2, vagas=2)),
    ("Apartamento 120 m² · região Intermediária", dict(tipo_pt="Apartamento", regiao="Intermediária", area_m2=120, quartos=3, banheiros=2, vagas=2)),
    ("Apartamento 120 m² · região Periférica",    dict(tipo_pt="Apartamento", regiao="Periférica",    area_m2=120, quartos=3, banheiros=2, vagas=2)),
    ("Cobertura 250 m² · região Nobre",           dict(tipo_pt="Cobertura",   regiao="Nobre",         area_m2=250, quartos=4, banheiros=4, vagas=3)),
    ("Studio 30 m² · região Intermediária",       dict(tipo_pt="Studio",      regiao="Intermediária", area_m2=30,  quartos=1, banheiros=1, vagas=0)),
]
cenarios = pd.DataFrame([c for _, c in descricao], index=[n for n, _ in descricao])
cenarios["latitude"] = cenarios["regiao"].map(centro["latitude"])
cenarios["longitude"] = cenarios["regiao"].map(centro["longitude"])
for coluna in NUMERICAS:
    if coluna not in cenarios.columns:
        cenarios[coluna] = X[coluna].median()
cenarios = cenarios[X.columns]

resultado = cenarios.copy()
resultado["PREÇO PREVISTO (R$)"] = pipeline.predict(cenarios)
resultado["R$/m² previsto"] = resultado["PREÇO PREVISTO (R$)"] / resultado["area_m2"]
display(resultado.T.style.format(lambda v: f"{v:,.2f}" if isinstance(v, (int, float, np.floating)) else v))

### 4.5 Do modelo à decisão: imóveis acima ou abaixo do preço esperado
Cada imóvel é previsto por um modelo que **não o viu no treino** (`cross_val_predict`). Desvio positivo = anunciado acima do esperado; negativo = abaixo (possível oportunidade **ou erro de cadastro**). É uma lista para **revisão**, não uma conclusão. Para a decisão comercial, olhamos principalmente os imóveis **ainda não vendidos** (disponíveis) com preço bem abaixo do esperado.

In [ ]:
base_ml["preco_esperado"] = cross_val_predict(montar(random_forest()), X, y, cv=cv, n_jobs=1)
base_ml["desvio_%"] = (base_ml["preco"] / base_ml["preco_esperado"] - 1) * 100

colunas_mostrar = ["bairro", "regiao", "tipo_pt", "area_m2", "preco", "preco_esperado", "desvio_%", "vendido"]
estilo = {"area_m2": "{:,.0f}", "preco": "R$ {:,.0f}", "preco_esperado": "R$ {:,.0f}", "desvio_%": "{:+.0f}%"}
print("Anunciados ACIMA do esperado (top 5):")
display(base_ml.sort_values("desvio_%", ascending=False)[colunas_mostrar].head(5).style.format(estilo).background_gradient(subset=["desvio_%"], cmap="Reds"))
print("Anunciados ABAIXO do esperado (top 5):")
display(base_ml.sort_values("desvio_%")[colunas_mostrar].head(5).style.format(estilo).background_gradient(subset=["desvio_%"], cmap="Blues_r"))

fora = int((base_ml["desvio_%"].abs() > 50).sum())
oportunidades = (~base_ml["vendido"]) & (base_ml["desvio_%"] <= -20)
RESULTADOS["fora_do_padrao"] = fora
RESULTADOS["oportunidades"] = int(oportunidades.sum())
print(f"{fora:,} de {len(base_ml):,} imóveis ({fora / len(base_ml):.0%}) ficam a mais de 50% do preço esperado: candidatos a revisão antes de negociar.")
print(f"Imóveis disponíveis (não vendidos) com preço 20% ou mais abaixo do esperado: {oportunidades.sum():,}.")
print("Por região:", (base_ml.loc[oportunidades, "regiao"].value_counts(normalize=True) * 100).round(0).astype(int).astype(str).add("%").to_dict())

<a id="5"></a>
## 5. Processamento distribuído: Apache Spark e o ecossistema Hadoop

### 5.1 O uso de Spark se justifica?
Com algumas dezenas de milhares de registros, **o pandas resolve tudo em segundos** numa única máquina, então o Spark não é necessário *hoje*. Ele passa a se justificar quando:

* o generator roda por meses (milhares de registros por dia, milhões por ano) ou a base é trocada por dados reais de portais;
* chegam fontes variadas (anúncios em texto, imagens, logs de acesso) que não cabem na memória de uma máquina;
* o processamento precisa ser paralelo e tolerante a falhas.

Por isso o mesmo tratamento é refeito abaixo com **Spark em modo local** (`local[*]`): o código é igual ao que rodaria num cluster; para escalar, mudam apenas o `master` (por exemplo, YARN) e o local dos arquivos (HDFS ou S3).

### 5.2 Papel dos componentes do ecossistema Hadoop

| Componente | Papel | Equivalente nesta solução | Num cluster real |
|---|---|---|---|
| **HDFS** | armazenamento distribuído em blocos replicados | arquivos CSV/Parquet no repositório e no disco | arquivos particionados por data no HDFS |
| **MapReduce** | processamento paralelo: map → shuffle → reduce | `map` + `reduceByKey` no Spark (seção 5.9) | jobs Spark (que substituem o MapReduce) |
| **Hive** | SQL sobre arquivos do HDFS | Spark SQL (seção 5.7) | Spark SQL com metastore do Hive |
| **YARN** | gerencia recursos e agenda os jobs do cluster | Spark `local[*]` | Spark com `master=yarn` |

Antes de usar o Spark, uma **simulação em Python puro** mostra a ideia por trás de HDFS e MapReduce: a base é dividida em blocos, cada bloco passa pela fase *map* e os resultados parciais são combinados na fase *reduce*.

In [ ]:
# ---------- Simulação didática: HDFS (blocos) + MapReduce em Python puro ----------
from collections import defaultdict

registros = list(zip(limpo["neighborhood"], limpo["price"]))
N_BLOCOS = 4                                                    # num HDFS real, blocos de 128 MB; aqui, 4 pedaços iguais
tam = -(-len(registros) // N_BLOCOS)
blocos = [registros[i * tam:(i + 1) * tam] for i in range(N_BLOCOS)]
print("Blocos (como o HDFS dividiria):", [len(b) for b in blocos], "registros")

def mapper(bloco):
    """MAP: cada registro vira um par (bairro, (1, preço))."""
    return [(bairro, (1, preco)) for bairro, preco in bloco]

def reducer(chave, valores):
    """REDUCE: soma as contagens e os preços de uma mesma chave."""
    n = sum(v[0] for v in valores)
    return chave, n, sum(v[1] for v in valores) / n

shuffle = defaultdict(list)                                     # SHUFFLE: agrupa pares de todos os blocos pela chave
for bloco in blocos:
    for chave, valor in mapper(bloco):
        shuffle[chave].append(valor)
saida = pd.DataFrame([reducer(k, v) for k, v in shuffle.items()], columns=["bairro", "imoveis", "preco_medio"])

confere = limpo.groupby("neighborhood")["price"].agg(["size", "mean"])
iguais = (saida.set_index("bairro").sort_index()["imoveis"].to_numpy() == confere.sort_index()["size"].to_numpy()).all()
print(f"MapReduce simulado == pandas (contagem por bairro): {iguais}")
display(saida.sort_values("imoveis", ascending=False).head(5).style.hide(axis="index")
        .format({"imoveis": "{:,.0f}", "preco_medio": "R$ {:,.0f}"}))

In [ ]:
# ---------- Simulação didática: Hive (SQL sobre arquivos) com SQLite da biblioteca padrão ----------
import sqlite3

con_hive = sqlite3.connect(":memory:")
limpo[["record_id", "property_type", "neighborhood", "regiao", "price", "area_m2"]].to_sql("imoveis_hive", con_hive, index=False)
hive = pd.read_sql("""
    SELECT regiao, COUNT(*) AS imoveis, ROUND(AVG(price / area_m2), 0) AS preco_m2_medio
    FROM imoveis_hive GROUP BY regiao ORDER BY preco_m2_medio DESC""", con_hive)
con_hive.close()
display(hive.style.hide(axis="index").format({"imoveis": "{:,.0f}", "preco_m2_medio": "R$ {:,.0f}"}))
print("O Hive faz o mesmo trabalho em escala: a consulta SQL vira jobs paralelos sobre arquivos do HDFS.")

### 5.3 Iniciando o Spark
Como na aula: criamos a **SparkSession** (ponto de entrada do Spark). O modo `local[*]` usa todos os núcleos da máquina; as configurações abaixo são as do notebook original do grupo (memória do *driver* pela variável `SPARK_DRIVER_MEMORY`, 8 partições de *shuffle* e interface web desligada).

In [ ]:
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .master("local[*]")
         .appName("PI-BigData-Imoveis")
         .config("spark.driver.memory", os.getenv("SPARK_DRIVER_MEMORY", "1g"))
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.enabled", "false")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| núcleos disponíveis:", spark.sparkContext.defaultParallelism)

### 5.4 Carregando os dados e tratando com DataFrames
Lemos **todos os CSVs brutos de uma só vez** (o Spark aceita uma lista de arquivos e processa cada pedaço em paralelo) e aplicamos, com `withColumn`, `filter` e `dropDuplicates`, **as mesmas regras de limpeza** da seção 2. Se o tratamento estiver certo, o total deve bater com o do pandas e com os parquets gerados pelo R.

In [ ]:
caminhos_csv = [baixar("data/raw", n) for n in arquivos_raw]
sdf_bruto = (spark.read.option("header", True).option("inferSchema", True)
             .option("multiLine", True).option("escape", '"')            # buyer_notes tem vírgulas e aspas
             .csv(caminhos_csv))
print(f"{sdf_bruto.count():,} registros lidos de {len(caminhos_csv)} CSVs em {sdf_bruto.rdd.getNumPartitions()} partições")
sdf_bruto.printSchema()
sdf_bruto.show(3, truncate=25)

In [ ]:
# Região de cada bairro (do generator.py) como DataFrame pequeno: o join usa broadcast, sem embaralhar a tabela grande
regioes = spark.createDataFrame(list(REGIAO_DO_BAIRRO.items()), ["neighborhood", "regiao"])

sdf = (sdf_bruto
       .dropDuplicates(["record_id"])                                              # duplicados
       .withColumn("state", F.lit("SP"))                                           # padroniza o estado
       .withColumn("sale_date", F.to_date(F.col("sale_date").cast("string")))      # texto -> data (inválida vira nula)
       .filter(F.col("price").isNotNull() & (F.col("price") > 0))                  # preço válido
       .filter(F.col("area_m2").isNotNull() & (F.col("area_m2") > 0))              # área válida
       .filter(F.col("bedrooms").isNull() | F.col("bedrooms").between(0, 20))      # quartos plausíveis
       .withColumn("preco_m2", F.col("price") / F.col("area_m2"))                  # indicador derivado
       .withColumn("vendido", F.col("sale_date").isNotNull())
       .withColumn("faixa_area", F.when(F.col("area_m2") <= 40, "até 40 m²")
                                  .when(F.col("area_m2") <= 80, "41 a 80 m²")
                                  .when(F.col("area_m2") <= 120, "81 a 120 m²")
                                  .when(F.col("area_m2") <= 200, "121 a 200 m²")
                                  .otherwise("acima de 200 m²"))
       .join(F.broadcast(regioes), "neighborhood", "left"))
sdf.cache()                                                                        # reaproveitado nas próximas seções

n_spark = sdf.count()
n_parquet = spark.read.parquet(*[baixar("data/processed", n) for n in arquivos_processed]).count()
RESULTADOS["registros_spark"] = n_spark
print(f"Spark (CSV bruto limpo): {n_spark:,}")
print(f"pandas (seção 2)        : {len(limpo):,}")
print(f"Parquets do R           : {n_parquet:,}")
print("Os três caminhos batem:", n_spark == len(limpo) == n_parquet)
sdf.select("neighborhood", "regiao", "property_type", "area_m2", "faixa_area", "price", "preco_m2", "vendido").show(5)

### 5.5 Seleção, filtros e ordenação
`select` escolhe colunas, `filter`/`where` escolhe linhas e `orderBy` ordena. Como no pandas, mas o Spark só executa quando uma **ação** (`show`, `count`, `collect`) é chamada: antes disso ele apenas monta o plano.

In [ ]:
print("Apartamentos da região Nobre com mais de 120 m², do mais caro para o mais barato:")
(sdf.select("neighborhood", "property_type", "area_m2", "price", "preco_m2")
    .filter((F.col("property_type") == "apartment") & (F.col("regiao") == "Nobre") & (F.col("area_m2") > 120))
    .orderBy(F.desc("price"))
    .show(5))

nobre_ap = sdf.filter((F.col("property_type") == "apartment") & (F.col("regiao") == "Nobre") & (F.col("area_m2") > 120)).count()
pandas_ap = int(((limpo["property_type"] == "apartment") & (limpo["regiao"] == "Nobre") & (limpo["area_m2"] > 120)).sum())
print(f"Quantidade pelo Spark: {nobre_ap:,} | pelo pandas: {pandas_ap:,} | iguais: {nobre_ap == pandas_ap}")

print("\nImóveis ainda não vendidos (sale_date nula) por região:")
sdf.filter(~F.col("vendido")).groupBy("regiao").count().orderBy(F.desc("count")).show()

### 5.6 Agrupamentos (`groupBy`) e indicadores
Cada `groupBy` + `agg` é o equivalente distribuído de um `GROUP BY`. Cada partição calcula um resultado parcial e o Spark combina os parciais (*shuffle*). Os resultados são conferidos contra o pandas e contra o Data Warehouse (`imoveis`, seção 4).

In [ ]:
por_tipo = (sdf.groupBy("property_type")
               .agg(F.count("*").alias("imoveis"),
                    F.round(F.avg("price"), 0).alias("preco_medio"),
                    F.round(F.expr("percentile_approx(price, 0.5)"), 0).alias("preco_mediano"),
                    F.round(F.avg("preco_m2"), 0).alias("preco_m2_medio"),
                    F.round(F.avg(F.col("vendido").cast("int")) * 100, 1).alias("pct_vendido"))
               .orderBy(F.desc("preco_m2_medio")))
por_tipo.show()

por_regiao_faixa = (sdf.groupBy("regiao").pivot("faixa_area", ["até 40 m²", "41 a 80 m²", "81 a 120 m²", "121 a 200 m²", "acima de 200 m²"])
                       .agg(F.round(F.avg("preco_m2"), 0)).orderBy("regiao"))
print("Preço médio por m² por região e faixa de área:")
por_regiao_faixa.show(truncate=False)

# ---------- Validação cruzada: Spark x pandas x Data Warehouse ----------
spark_tipo = por_tipo.toPandas().set_index("property_type")
pandas_tipo = limpo.groupby("property_type").agg(imoveis=("price", "size"), preco_m2_medio=("preco_m2", "mean"))
dw_tipo = imoveis.groupby("tipo").size()
mesmas_contagens = (spark_tipo["imoveis"].sort_index() == pandas_tipo["imoveis"].sort_index()).all()
mesmos_precos = np.allclose(spark_tipo["preco_m2_medio"].sort_index(), pandas_tipo["preco_m2_medio"].sort_index().round(0), atol=1)
bate_dw = (spark_tipo["imoveis"].sort_index() == dw_tipo.sort_index()).all()
print(f"Contagem por tipo: Spark == pandas: {mesmas_contagens} | Spark == DW: {bate_dw}")
print(f"Preço médio por m² por tipo: Spark == pandas (tolerância de R$ 1): {mesmos_precos}")

### 5.7 Spark SQL (o papel do Hive)
`createOrReplaceTempView` registra o DataFrame como uma tabela temporária e permite escrever **SQL puro**. O resultado é um DataFrame, igual ao do `groupBy`: por trás, o mesmo otimizador (Catalyst) gera o plano.

In [ ]:
sdf.createOrReplaceTempView("imoveis_spark")

print("1) Tipo × região, apenas combinações com pelo menos 30 imóveis (HAVING):")
spark.sql("""
    SELECT property_type, regiao,
           COUNT(*)                       AS imoveis,
           ROUND(AVG(price), 0)           AS preco_medio,
           ROUND(AVG(preco_m2), 0)        AS preco_m2_medio
    FROM imoveis_spark
    GROUP BY property_type, regiao
    HAVING COUNT(*) >= 30
    ORDER BY property_type, preco_m2_medio DESC
""").show(12)

print("2) Os 3 bairros mais caros por m² em cada tipo de imóvel (função de janela RANK):")
spark.sql("""
    SELECT property_type, neighborhood, imoveis, preco_m2_medio, posicao
    FROM (
        SELECT property_type, neighborhood, COUNT(*) AS imoveis, ROUND(AVG(preco_m2), 0) AS preco_m2_medio,
               RANK() OVER (PARTITION BY property_type ORDER BY AVG(preco_m2) DESC) AS posicao
        FROM imoveis_spark
        GROUP BY property_type, neighborhood
        HAVING COUNT(*) >= 30
    ) ranking
    WHERE posicao <= 3
    ORDER BY property_type, posicao
""").show(20, truncate=False)

print("3) Quanto cada região está acima ou abaixo da média geral de preço por m²:")
comparacao_regiao = spark.sql("""
    SELECT r.regiao, r.imoveis, r.preco_m2_medio, g.media_geral,
           ROUND((r.preco_m2_medio / g.media_geral - 1) * 100, 1) AS desvio_pct
    FROM (SELECT regiao, COUNT(*) AS imoveis, AVG(preco_m2) AS preco_m2_medio FROM imoveis_spark GROUP BY regiao) r
    CROSS JOIN (SELECT AVG(preco_m2) AS media_geral FROM imoveis_spark) g
    ORDER BY r.preco_m2_medio DESC
""").toPandas()
display(comparacao_regiao.style.hide(axis="index")
        .format({"imoveis": "{:,.0f}", "preco_m2_medio": "R$ {:,.0f}", "media_geral": "R$ {:,.0f}", "desvio_pct": "{:+.1f}%"})
        .background_gradient(subset=["desvio_pct"], cmap="RdYlGn"))

### 5.8 Partições e plano de execução
O Spark divide os dados em **partições**, e cada núcleo processa uma por vez: é o que torna o processamento distribuído. `repartition` redistribui os dados; `explain()` mostra o plano físico que o Spark escolheu (o `BroadcastHashJoin` da seção 5.4 e o `HashAggregate` em duas fases, parcial e final, são visíveis).

In [ ]:
print("Partições do DataFrame limpo:", sdf.rdd.getNumPartitions())
dist = (sdf.repartition(4).withColumn("particao", F.spark_partition_id())
           .groupBy("particao").count().orderBy("particao").toPandas())
display(dist.style.hide(axis="index").format({"count": "{:,.0f}"}).bar(subset=["count"], color="#5fa8d3"))

print("\nPlano físico de execução (preço médio por região):")
sdf.groupBy("regiao").agg(F.avg("preco_m2")).explain()

### 5.9 MapReduce no Spark (RDD)
É o MapReduce clássico em duas linhas: `map` transforma cada imóvel em `(bairro, 1)`, o *shuffle* agrupa por chave e `reduceByKey` soma. Os caminhos (RDD, DataFrame, pandas e Data Warehouse) precisam dar o mesmo resultado; o teste automático do repositório confere estas duas linhas.

In [ ]:
from operator import add

mapreduce = dict(sdf.select("neighborhood").rdd.map(lambda linha: (linha[0], 1)).reduceByKey(add).collect())
dataframe = {r["neighborhood"]: r["count"] for r in sdf.groupBy("neighborhood").count().collect()}
pandas_bairro = limpo.groupby("neighborhood").size().to_dict()
dw_bairro = imoveis.groupby("bairro").size().to_dict()
print("MapReduce == DataFrame Spark:", mapreduce == dataframe)
print("Spark == pandas/DW          :", mapreduce == pandas_bairro == dw_bairro)      # pandas (seção 2) e Data Warehouse (seção 3)
display(pd.Series(mapreduce).sort_values(ascending=False).head(5).rename("imóveis (MapReduce)").to_frame())

### 5.10 Teste de escala: replicando a base
Para ver o Spark trabalhando com um volume maior, a base limpa é **replicada** (`crossJoin` com `spark.range`) até chegar perto de 2 milhões de linhas, e o mesmo `groupBy` é medido nos dois tamanhos. É um teste de **comportamento**: os dados replicados não trazem informação nova, e o tempo depende da máquina (no Colab gratuito costuma ser de poucos núcleos).

In [ ]:
import time

def medir(tabela):
    """Tempo de um groupBy com agregações sobre a tabela (média de 2 execuções, em segundos)."""
    tempos = []
    for _ in range(2):
        ini = time.perf_counter()
        tabela.groupBy("neighborhood", "property_type").agg(F.count("*"), F.avg("price"), F.avg("preco_m2")).collect()
        tempos.append(time.perf_counter() - ini)
    return sum(tempos) / len(tempos)

REPLICAS = max(1, 2_000_000 // n_spark)
sdf_grande = sdf.crossJoin(spark.range(REPLICAS).withColumnRenamed("id", "replica"))
escala = pd.DataFrame({"base": ["original", f"replicada ({REPLICAS}×)"],
                       "linhas": [n_spark, sdf_grande.count()],
                       "segundos (groupBy)": [medir(sdf), medir(sdf_grande)]})
display(escala.style.hide(axis="index").format({"linhas": "{:,.0f}", "segundos (groupBy)": "{:.2f}"}))
fator_linhas = escala["linhas"].iloc[1] / escala["linhas"].iloc[0]
fator_tempo = escala["segundos (groupBy)"].iloc[1] / escala["segundos (groupBy)"].iloc[0]
RESULTADOS.update(spark_fator_linhas=fator_linhas, spark_fator_tempo=fator_tempo)
print(f"Interpretação: as linhas cresceram {fator_linhas:.0f}× e o tempo, {fator_tempo:.1f}×. "
      + ("O tempo cresce menos que o volume porque parte do custo é fixo (planejamento, início das tarefas) e o paralelismo é mais bem aproveitado com mais dados."
         if fator_tempo < fator_linhas else
         "Nesta máquina o tempo acompanha o volume; num cluster, mais nós dividiriam o trabalho."))

### 5.11 Gravando o resultado em Parquet e encerrando o Spark
**Parquet** é o formato colunar usado no ecossistema Hadoop: comprime bem e lê só as colunas necessárias. Gravamos a base tratada **sem os dados pessoais** (nome do vendedor, identificação e observações do comprador), **particionada por tipo de imóvel** (uma pasta por tipo, como seria no HDFS), e lemos de volta para conferir.

In [ ]:
destino = os.path.join(PASTA_DADOS, "saida_spark", "imoveis_limpos")
(sdf.drop("seller_name", "buyer_id", "buyer_notes")
    .write.mode("overwrite").partitionBy("property_type").parquet(destino))

conferido = spark.read.parquet(destino)
print("Pastas geradas:", sorted(p for p in os.listdir(destino) if p.startswith("property_type=")))
print(f"Registros gravados e relidos: {conferido.count():,} | iguais ao DataFrame limpo: {conferido.count() == n_spark}")
print("Colunas pessoais ausentes:", not {"seller_name", "buyer_id", "buyer_notes"} & set(conferido.columns))

In [ ]:
# Encerrando o Spark (libera memória e núcleos)
spark.stop()
print("Spark encerrado.")

### 5.12 Como a solução escala
* **Armazenamento:** os CSVs e parquets passariam para HDFS (ou S3), particionados por data de carga; o Data Warehouse poderia migrar para um engine colunar (Hive, Spark SQL sobre Delta/Iceberg) quando o PostgreSQL deixar de dar conta.
* **Processamento:** o mesmo código Spark roda num cluster trocando `local[*]` por `yarn`; o número de partições cresce com os dados e cada nó processa só os seus blocos.
* **Ingestão:** o generator hoje publica via Git; em escala, os lotes iriam para uma fila (Kafka) e seriam processados por Spark Structured Streaming.

<a id="6"></a>
## 6. Análise final e insights
Esta seção junta os resultados das etapas anteriores. Os números e as frases abaixo são **calculados a partir dos dados carregados** (nada digitado à mão), então continuam corretos quando o generator publicar novos lotes.

### 6.1 Desempenho por corretor
O Data Warehouse não guarda o corretor (só o `seller_id`, um código), então esta tabela sai da base tratada da seção 2 (`limpo`). O **nome** do corretor não é usado (LGPD). Pergunta: algum corretor vende mais ou vende imóveis mais caros? A taxa de venda é comparada com o que o **acaso** explicaria: com poucos imóveis, a taxa oscila bastante mesmo que todos tenham o mesmo desempenho.

In [ ]:
from scipy.stats import chi2_contingency

corretores = limpo.groupby("seller_id").agg(imoveis=("price", "size"), vendidos=("vendido", "sum"),
                                            preco_m2_mediano=("preco_m2", "median"))
corretores["taxa_venda_%"] = corretores["vendidos"] / corretores["imoveis"] * 100
corretores["valor_vendido_mi"] = limpo[limpo["vendido"]].groupby("seller_id")["price"].sum() / 1e6
corretores["ticket_medio_vendido"] = limpo[limpo["vendido"]].groupby("seller_id")["price"].mean()
taxa_geral = limpo["vendido"].mean()
corretores["z"] = (corretores["taxa_venda_%"] / 100 - taxa_geral) / np.sqrt(taxa_geral * (1 - taxa_geral) / corretores["imoveis"])
corretores = corretores.sort_values("valor_vendido_mi", ascending=False)

p_valor = chi2_contingency(np.array([corretores["vendidos"], corretores["imoveis"] - corretores["vendidos"]]))[1]
RESULTADOS.update(n_corretores=len(corretores), taxa_corretor_min=corretores["taxa_venda_%"].min(),
                  taxa_corretor_max=corretores["taxa_venda_%"].max(), p_corretores=p_valor)

mostrar = corretores.head(15)
display(mostrar.drop(columns="z").style.format({"imoveis": "{:,.0f}", "vendidos": "{:,.0f}", "preco_m2_mediano": "R$ {:,.0f}",
        "taxa_venda_%": "{:.1f}%", "valor_vendido_mi": "R$ {:,.1f} mi", "ticket_medio_vendido": "R$ {:,.0f}"})
        .bar(subset=["valor_vendido_mi"], color="#5fa8d3").background_gradient(subset=["taxa_venda_%"], cmap="RdYlGn"))

fig, ax = plt.subplots(1, 2, figsize=(13, max(3.5, 0.3 * len(mostrar))))
ordem = mostrar.sort_values("taxa_venda_%").index
ax[0].barh(ordem, mostrar.loc[ordem, "taxa_venda_%"], color="#81b29a"); ax[0].axvline(taxa_geral * 100, color="r", ls="--", label="média geral")
ax[0].set_title("Taxa de venda por corretor (%)"); ax[0].set_xlim(0, 100); ax[0].legend(loc="lower right")
ordem = mostrar.sort_values("valor_vendido_mi").index
ax[1].barh(ordem, mostrar.loc[ordem, "valor_vendido_mi"], color="#3d405b"); ax[1].set_title("Valor vendido por corretor (R$ milhões)")
plt.tight_layout(); plt.show()

fora = corretores[corretores["z"].abs() > 2]
print(f"Interpretação: {len(corretores)} corretores; a taxa de venda vai de {corretores['taxa_venda_%'].min():.1f}% a {corretores['taxa_venda_%'].max():.1f}% "
      f"(média geral {taxa_geral:.1%}). Teste qui-quadrado entre corretores: p = {p_valor:.2f}. "
      + (f"As diferenças são maiores do que o acaso explicaria; destaque para {', '.join(fora.index)}." if p_valor < 0.05 and len(fora) else
         "As diferenças são compatíveis com o acaso: não há evidência de que algum corretor venda melhor. Isso é esperado numa base sintética, em que o corretor é sorteado."))

### 6.2 Taxa de venda por segmento
Onde o estoque vende e onde fica parado? A taxa de venda (% dos imóveis com data de venda) é cruzada por **tipo × região** e por **faixa de área × região**, a partir do Data Warehouse.

In [ ]:
FAIXAS = ["até 40 m²", "41 a 80 m²", "81 a 120 m²", "121 a 200 m²", "acima de 200 m²"]
tipo_regiao = imoveis.pivot_table(index="tipo_pt", columns="regiao", values="vendido", aggfunc="mean")[ordem_regioes] * 100
faixa_regiao = imoveis.pivot_table(index="faixa_area", columns="regiao", values="vendido", aggfunc="mean").reindex(FAIXAS)[ordem_regioes] * 100
n_tipo_regiao = imoveis.pivot_table(index="tipo_pt", columns="regiao", values="vendido", aggfunc="size")[ordem_regioes]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.heatmap(tipo_regiao, annot=True, fmt=".0f", cmap="RdYlGn", vmin=50, vmax=90, ax=ax[0], cbar_kws={"label": "% vendido"}); ax[0].set_title("Taxa de venda: tipo × região (%)"); ax[0].set_ylabel("")
sns.heatmap(faixa_regiao, annot=True, fmt=".0f", cmap="RdYlGn", vmin=50, vmax=90, ax=ax[1], cbar_kws={"label": "% vendido"}); ax[1].set_title("Taxa de venda: faixa de área × região (%)"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

# ---------- Diferença real ou acaso? ----------
seg = imoveis.assign(segmento=imoveis["tipo_pt"] + " · " + imoveis["regiao"])
tab = seg.groupby("segmento")["vendido"].agg(["sum", "size"])
p_seg = chi2_contingency(np.array([tab["sum"], tab["size"] - tab["sum"]]))[1]
taxas = tab["sum"] / tab["size"] * 100
menor_n = int(tab["size"].min())
ruido = 2 * np.sqrt(imoveis["vendido"].mean() * (1 - imoveis["vendido"].mean()) / menor_n) * 100
RESULTADOS.update(seg_melhor=f"{taxas.idxmax()} ({taxas.max():.0f}%)", seg_pior=f"{taxas.idxmin()} ({taxas.min():.0f}%)", p_segmentos=p_seg)
display(taxas.sort_values(ascending=False).rename("taxa de venda (%)").to_frame().join(tab["size"].rename("imóveis"))
        .style.format({"taxa de venda (%)": "{:.1f}%", "imóveis": "{:,.0f}"}).background_gradient(subset=["taxa de venda (%)"], cmap="RdYlGn"))
print("As cores usam escala fixa de 50% a 90%: assim, diferenças pequenas não parecem grandes.")
print(f"Interpretação: a maior taxa de venda é de {RESULTADOS['seg_melhor']} e a menor, de {RESULTADOS['seg_pior']}. "
      f"Com no mínimo {menor_n:,} imóveis por segmento, o acaso explicaria variações de cerca de ±{ruido:.1f} pontos percentuais. "
      f"Teste qui-quadrado entre os {len(tab)} segmentos: p = {p_seg:.2f}. "
      + ("Há diferença real entre segmentos: vale priorizar a divulgação dos que vendem menos." if p_seg < 0.05 else
         "Não há diferença que o acaso não explique: o estoque parado se distribui de forma parecida entre os segmentos (reflexo do generator, que sorteia a venda sem olhar tipo ou região)."))

### 6.3 Previsto × real por bairro
Onde o modelo da seção 4.2 erra mais? Usamos o preço esperado da seção 4.5 (`cross_val_predict`: cada imóvel é previsto por um modelo que não o viu) e agrupamos o erro por bairro. **MAPE** é o erro percentual médio em módulo; o **viés** é a mediana do desvio com sinal (positivo = o preço anunciado costuma ficar acima do previsto).

In [ ]:
base_ml["erro_abs_%"] = base_ml["desvio_%"].abs()
por_bairro_ml = base_ml.groupby("bairro").agg(regiao=("regiao", "first"), imoveis=("preco", "size"), preco_real_mediano=("preco", "median"),
                                              preco_previsto_mediano=("preco_esperado", "median"), MAPE=("erro_abs_%", "mean"),
                                              vies=("desvio_%", "median")).sort_values("MAPE", ascending=False)
mape_regiao = base_ml.groupby("regiao")["erro_abs_%"].mean().loc[ordem_regioes]
RESULTADOS.update(bairro_pior_mape=f"{por_bairro_ml.index[0]} ({por_bairro_ml['MAPE'].iloc[0]:.0f}%)",
                  bairro_melhor_mape=f"{por_bairro_ml.index[-1]} ({por_bairro_ml['MAPE'].iloc[-1]:.0f}%)")

estilo = {"imoveis": "{:,.0f}", "preco_real_mediano": "R$ {:,.0f}", "preco_previsto_mediano": "R$ {:,.0f}", "MAPE": "{:.1f}%", "vies": "{:+.1f}%"}
print("10 bairros com maior erro:")
display(por_bairro_ml.head(10).style.format(estilo).background_gradient(subset=["MAPE"], cmap="Reds"))
print("5 bairros com menor erro:")
display(por_bairro_ml.tail(5).style.format(estilo).background_gradient(subset=["MAPE"], cmap="Greens_r"))

cores = dict(zip(ordem_regioes, sns.color_palette("deep", 3)))
graf = por_bairro_ml.sort_values("MAPE")
fig, ax = plt.subplots(figsize=(8, 0.22 * len(graf) + 1.5))
ax.barh(graf.index, graf["MAPE"], color=[cores[r] for r in graf["regiao"]])
ax.axvline(base_ml["erro_abs_%"].mean(), color="k", ls="--", lw=1, label=f"média geral ({base_ml['erro_abs_%'].mean():.0f}%)")
ax.set_xlabel("MAPE (%)"); ax.set_xlim(0, graf["MAPE"].max() * 1.45); ax.set_title("Erro do modelo por bairro (cores = região)")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in cores.values()] + ax.get_legend_handles_labels()[0],
          labels=list(cores) + [f"média geral ({base_ml['erro_abs_%'].mean():.0f}%)"], loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

print(f"Interpretação: o erro médio vai de {RESULTADOS['bairro_melhor_mape']} (melhor) a {RESULTADOS['bairro_pior_mape']} (pior). "
      f"Por região: " + "; ".join(f"{r} {v:.0f}%" for r, v in mape_regiao.items()) + ". "
      f"O viés fica entre {por_bairro_ml['vies'].min():+.1f}% e {por_bairro_ml['vies'].max():+.1f}%: "
      + ("o modelo não favorece nenhum bairro de forma sistemática (o erro se distribui nos dois sentidos)."
         if por_bairro_ml["vies"].abs().max() < 5 else "alguns bairros têm viés relevante e merecem atenção (por exemplo, incluir o bairro como variável)."))

### 6.4 Mapa de calor por bairro (interativo)
Onde o m² é mais caro na cidade? O mapa abaixo é **interativo** (`folium`): cada **bairro** é um círculo na sua posição (mediana de latitude e longitude), com **cor** = preço mediano por m² (do amarelo ao vermelho-escuro) e **tamanho** = quantidade de imóveis. Passe o mouse para ver os números, clique para o detalhe e use o seletor no canto para ligar ou desligar cada **região**. O mapa também é salvo em HTML para abrir na apresentação, sem o notebook.

> Se a biblioteca `folium` não estiver instalada (ela está no `requirements-notebook.txt`), a célula mostra o gráfico estático equivalente.

In [ ]:
centro_bairro = imoveis.groupby("bairro").agg(lat=("latitude", "median"), lon=("longitude", "median"), preco_m2=("preco_m2", "median"),
                                              preco=("preco", "median"), imoveis=("preco", "size"), pct_vendido=("vendido", "mean"),
                                              regiao=("regiao", "first"))

def mapa_estatico():
    # Plano B: bolhas por bairro e grade de preço/m² em gráficos do matplotlib
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    sc = ax[0].scatter(centro_bairro["lon"], centro_bairro["lat"], c=centro_bairro["preco_m2"], s=centro_bairro["imoveis"] / centro_bairro["imoveis"].max() * 600 + 40,
                       cmap="YlOrRd", edgecolor="k", linewidth=.4, alpha=.9)
    plt.colorbar(sc, ax=ax[0], label="preço mediano por m² (R$)")
    for nome in list(centro_bairro["preco_m2"].nlargest(4).index) + list(centro_bairro["preco_m2"].nsmallest(3).index):
        ax[0].annotate(nome, (centro_bairro.loc[nome, "lon"], centro_bairro.loc[nome, "lat"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax[0].set_title("Bairros: preço/m² mediano (tamanho = nº de imóveis)")
    hb = ax[1].hexbin(imoveis["longitude"], imoveis["latitude"], C=imoveis["preco_m2"], reduce_C_function=np.median, gridsize=35, mincnt=5, cmap="YlOrRd")
    plt.colorbar(hb, ax=ax[1], label="preço mediano por m² (R$)"); ax[1].set_title("Grade sobre o mapa: preço/m² mediano")
    for a in ax:
        a.set_xlabel("longitude"); a.set_ylabel("latitude"); a.set_aspect(1 / np.cos(np.radians(imoveis["latitude"].mean())))
    plt.tight_layout(); plt.show()

try:
    import folium
    from branca.colormap import LinearColormap
except ImportError:
    print("folium não instalado: mostrando o gráfico estático (para o mapa interativo, instale com: pip install folium).")
    mapa_estatico()
else:
    escala = LinearColormap(["#ffffb2", "#fd8d3c", "#bd0026", "#67001f"], vmin=float(centro_bairro["preco_m2"].min()),
                            vmax=float(centro_bairro["preco_m2"].max()), caption="Preço mediano por m² (R$)")
    mapa = folium.Map(location=[centro_bairro["lat"].mean(), centro_bairro["lon"].mean()], zoom_start=11, tiles="CartoDB positron")
    camadas = {r: folium.FeatureGroup(name=f"Região {r}", show=True) for r in ordem_regioes}
    for nome, b in centro_bairro.iterrows():
        detalhe = (f"<b>{nome}</b> ({b['regiao']})<br>Preço/m² mediano: {brl(b['preco_m2'])}<br>Preço mediano: {brl(b['preco'])}"
                   f"<br>Imóveis: {b['imoveis']:,.0f}<br>Vendidos: {b['pct_vendido']:.0%}")
        folium.CircleMarker(location=[b["lat"], b["lon"]], radius=6 + 14 * b["imoveis"] / centro_bairro["imoveis"].max(),
                            color="#333333", weight=1, fill=True, fill_color=escala(b["preco_m2"]), fill_opacity=0.85,
                            tooltip=f"{nome}: {brl(b['preco_m2'])}/m²", popup=folium.Popup(detalhe, max_width=260)).add_to(camadas[b["regiao"]])
    for camada in camadas.values():
        camada.add_to(mapa)
    escala.add_to(mapa)
    folium.LayerControl(collapsed=False).add_to(mapa)
    mapa.fit_bounds([[centro_bairro["lat"].min(), centro_bairro["lon"].min()], [centro_bairro["lat"].max(), centro_bairro["lon"].max()]])
    arquivo_mapa = os.path.join(PASTA_DADOS, "mapa_bairros.html")
    mapa.save(arquivo_mapa)
    print(f"Mapa salvo em: {arquivo_mapa} (abra no navegador para a apresentação).")
    display(mapa)

pos = imoveis.groupby("regiao")[["latitude", "longitude"]].mean()
dlat = (pos.loc["Nobre", "latitude"] - pos.loc["Periférica", "latitude"]) * 111
dlon = (pos.loc["Nobre", "longitude"] - pos.loc["Periférica", "longitude"]) * 111 * np.cos(np.radians(imoveis["latitude"].mean()))
RESULTADOS["m2_bairro_top"] = centro_bairro["preco_m2"].idxmax()
print(f"Interpretação: o bairro de maior preço/m² mediano é {centro_bairro['preco_m2'].idxmax()} ({brl(centro_bairro['preco_m2'].max())}) e o de menor é "
      f"{centro_bairro['preco_m2'].idxmin()} ({brl(centro_bairro['preco_m2'].min())}). Em média, a região Nobre fica {abs(dlat):.0f} km ao "
      f"{'sul' if dlat < 0 else 'norte'} e {abs(dlon):.0f} km a {'oeste' if dlon < 0 else 'leste'} da Periférica. "
      "No mapa, as manchas mais escuras indicam as áreas de maior valor; é essa relação entre posição e preço que latitude e longitude permitem ao modelo aprender.")

### 6.5 Ticket por faixa de área × faixa de idade
Imóvel mais novo vale mais? O preço **total** cresce com a área quase por definição, então a **idade** é melhor medida pelo **preço por m²**. Para não confundir com a região (que domina o preço do m²), comparamos também o preço/m² **relativo à mediana da própria região**: 1,00 = igual ao típico da região. A idade vem de `faixa_idade` do DW (calculada a partir do ano de construção).

In [ ]:
from scipy.stats import spearmanr

FAIXAS_AREA = ["até 40 m²", "41 a 80 m²", "81 a 120 m²", "121 a 200 m²", "acima de 200 m²"]
FAIXAS_IDADE = ["até 10 anos", "11 a 20 anos", "21 a 30 anos", "31 a 40 anos", "mais de 40 anos"]

ti = imoveis[imoveis["faixa_idade"].isin(FAIXAS_IDADE)].copy()                  # descarta "Desconhecida", se houver
ti["preco_m2_relativo"] = ti["preco_m2"] / ti.groupby("regiao")["preco_m2"].transform("median")

def cruzar(coluna, funcao="median"):
    return ti.pivot_table(index="faixa_area", columns="faixa_idade", values=coluna, aggfunc=funcao).reindex(index=FAIXAS_AREA, columns=FAIXAS_IDADE)

ticket = cruzar("preco")
ticket_m2 = cruzar("preco_m2")
n_celula = cruzar("preco", "size")
print("Preço mediano (R$) por faixa de área × faixa de idade:")
display(ticket.style.format("R$ {:,.0f}").background_gradient(cmap="YlOrRd", axis=None))
print("Quantidade de imóveis em cada célula:")
display(n_celula.style.format("{:,.0f}").background_gradient(cmap="Blues", axis=None))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
sns.heatmap(ticket / 1e6, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax[0], cbar_kws={"label": "R$ milhões"}); ax[0].set_title("Preço mediano (R$ mi)")
sns.heatmap(ticket_m2, annot=True, fmt=",.0f", cmap="YlOrRd", vmin=0, ax=ax[1], cbar_kws={"label": "R$ por m²"}); ax[1].set_title("Preço mediano por m² (R$)")
for a in ax:
    a.set_xlabel("faixa de idade"); a.set_ylabel("faixa de área")
plt.tight_layout(); plt.show()

# ---------- A idade importa? (preço/m² relativo à região) ----------
por_idade = ti.groupby("faixa_idade").agg(imoveis=("preco", "size"), preco_m2_mediano=("preco_m2", "median"),
                                          relativo_a_regiao=("preco_m2_relativo", "median")).reindex(FAIXAS_IDADE)
display(por_idade.style.format({"imoveis": "{:,.0f}", "preco_m2_mediano": "R$ {:,.0f}", "relativo_a_regiao": "{:.3f}"})
        .background_gradient(subset=["relativo_a_regiao"], cmap="RdYlGn"))
rho_m2, p_m2 = spearmanr(ti["ano_construcao"], ti["preco_m2"], nan_policy="omit")
rho_rel, p_rel = spearmanr(ti["ano_construcao"], ti["preco_m2_relativo"], nan_policy="omit")
dif_idade = por_idade["relativo_a_regiao"].iloc[0] / por_idade["relativo_a_regiao"].iloc[-1] - 1
razao_area = ticket.median(axis=1).iloc[-1] / ticket.median(axis=1).iloc[0]
RESULTADOS.update(idade_rho=rho_rel, idade_p=p_rel, idade_dif=dif_idade, razao_area_preco=razao_area, menor_celula=int(n_celula.min().min()))
print(f"Interpretação: o preço total sobe com a área: a faixa \"{FAIXAS_AREA[-1]}\" tem preço mediano cerca de {razao_area:.1f}× o da faixa \"{FAIXAS_AREA[0]}\". "
      f"Quanto à idade, a faixa mais nova fica {dif_idade:+.1%} em relação à mais antiga no valor do m², descontada a região. "
      f"Correlação de Spearman entre ano de construção e preço/m²: {rho_m2:.3f} (p = {p_m2:.2f}); descontada a região: {rho_rel:.3f} (p = {p_rel:.2f}). "
      + ("A relação com a idade é desprezível (|ρ| < 0,10). Isso é esperado numa base sintética, em que o generator sorteia o ano de construção sem olhar o preço; em dados reais, imóveis novos tendem a valer mais."
         if abs(rho_rel) < 0.10 else "Há relação com a idade, que merece ser investigada (veja o tamanho do efeito na tabela por faixa de idade).")
      + f" A menor célula do cruzamento tem {RESULTADOS['menor_celula']:,} imóveis.")

### 6.6 Condomínio × preço
O condomínio pesa mais nos imóveis baratos? Duas medidas: o **condomínio por m²** e o **peso anual do condomínio no preço** (12 mensalidades ÷ preço do imóvel). Imóveis com condomínio **zerado** (nesta base, as casas) ficam fora das médias: "sem condomínio" não é "condomínio baixo", e incluí-los puxaria as medianas para zero.

In [ ]:
zero_por_tipo = imoveis.groupby("tipo_pt")["condominio"].apply(lambda s: (s == 0).mean()).rename("% sem condomínio")
display(zero_por_tipo.to_frame().style.format("{:.0%}").background_gradient(cmap="Blues"))

cd = imoveis[imoveis["condominio"] > 0].copy()
cd["condominio_m2"] = cd["condominio"] / cd["area_m2"]
cd["peso_anual_%"] = cd["condominio"] * 12 / cd["preco"] * 100
print(f"{len(imoveis) - len(cd):,} imóveis sem condomínio foram excluídos desta análise ({1 - len(cd) / len(imoveis):.0%} da base).")

por_tipo_cd = cd.groupby("tipo_pt").agg(imoveis=("preco", "size"), condominio_mediano=("condominio", "median"),
                                        condominio_m2=("condominio_m2", "median"), peso_anual=("peso_anual_%", "median"))
display(por_tipo_cd.style.format({"imoveis": "{:,.0f}", "condominio_mediano": "R$ {:,.0f}", "condominio_m2": "R$ {:,.1f}", "peso_anual": "{:.2f}%"})
        .background_gradient(subset=["peso_anual"], cmap="Reds"))

# ---------- Por faixa de preço (quartis): o condomínio pesa mais nos baratos? ----------
cd["faixa_preco"] = pd.qcut(cd["preco"], 4, labels=["25% mais baratos", "2º quartil", "3º quartil", "25% mais caros"])
por_preco = cd.groupby("faixa_preco", observed=True).agg(imoveis=("preco", "size"), preco_mediano=("preco", "median"),
                                                         condominio_mediano=("condominio", "median"), peso_anual=("peso_anual_%", "median"))
display(por_preco.style.format({"imoveis": "{:,.0f}", "preco_mediano": "R$ {:,.0f}", "condominio_mediano": "R$ {:,.0f}", "peso_anual": "{:.2f}%"})
        .background_gradient(subset=["peso_anual"], cmap="Reds"))

# ---------- Gráficos ----------
peso_tr = cd.pivot_table(index="tipo_pt", columns="regiao", values="peso_anual_%", aggfunc="median")[ordem_regioes]
amostra = cd.sample(min(3000, len(cd)), random_state=42)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
sns.heatmap(peso_tr, annot=True, fmt=".2f", cmap="Reds", ax=ax[0], cbar_kws={"label": "% do preço por ano"}); ax[0].set_title("Peso anual do condomínio no preço (mediana, %)"); ax[0].set_ylabel("")
ax[1].scatter(amostra["preco"] / 1e6, amostra["condominio"], s=8, alpha=.35, color="#3d405b")
ax[1].set_xscale("log"); ax[1].set_xlabel("Preço (R$ milhões, escala log)"); ax[1].set_ylabel("Condomínio mensal (R$)"); ax[1].set_title("Condomínio × preço (amostra de 3.000 imóveis)")
plt.tight_layout(); plt.show()

rho_cd, p_cd = spearmanr(cd["condominio"], cd["preco"])
peso_q1, peso_q4 = por_preco["peso_anual"].iloc[0], por_preco["peso_anual"].iloc[-1]
RESULTADOS.update(cond_rho=rho_cd, cond_p=p_cd, cond_peso_baratos=peso_q1, cond_peso_caros=peso_q4, cond_pct_zero=1 - len(cd) / len(imoveis))
cond_q1, cond_q4 = por_preco["condominio_mediano"].iloc[0], por_preco["condominio_mediano"].iloc[-1]
print(f"Interpretação: o condomínio mediano é de {brl(cd['condominio'].median())} por mês ({brl(cond_q1)} nos 25% mais baratos e {brl(cond_q4)} nos 25% mais caros). "
      f"O peso anual no preço vai de {peso_q1:.2f}% nos imóveis mais baratos a {peso_q4:.2f}% nos mais caros ({peso_q1 / peso_q4:.1f}× maior nos baratos). "
      f"Correlação de Spearman entre condomínio e preço: {rho_cd:.3f} (p = {p_cd:.2f}). "
      + ("O condomínio quase não muda com o preço; por isso ele pesa proporcionalmente muito mais nos imóveis baratos. Na base sintética o generator sorteia o condomínio de forma independente do preço; "
         "em dados reais, o condomínio tende a subir com o padrão do imóvel."
         if abs(rho_cd) < 0.10 else "O condomínio sobe com o preço, o que suaviza a diferença de peso entre imóveis baratos e caros.")
      + " Casas, sem condomínio, ficaram fora da análise.")

In [ ]:
# ---------- Indicadores de síntese ----------
from IPython.display import Markdown

regioes_m2 = imoveis.groupby("regiao")["preco_m2"].mean().loc[ordem_regioes]
tipos_m2 = imoveis.groupby("tipo_pt")["preco_m2"].mean().sort_values(ascending=False)
vendidos = imoveis[imoveis["vendido"]].copy()
vendidos["data_venda"] = pd.to_datetime(vendidos["data_venda"])
anual = vendidos.groupby(vendidos["data_venda"].dt.year)["preco_m2"].median()
anual = anual[vendidos.groupby(vendidos["data_venda"].dt.year).size() >= 100]       # só anos com volume mínimo
variacao_anual = (anual.max() / anual.min() - 1) if len(anual) > 1 else float("nan")
hoje = pd.Timestamp.today().normalize()
futuras = int((vendidos["data_venda"] > hoje).sum())

resumo = pd.Series({
    "Imóveis analisados (DW)": f"{len(imoveis):,}",
    "Registros brutos → válidos": f"{RESULTADOS['registros_brutos']:,} → {RESULTADOS['registros_validos']:,}",
    "Preço mediano": brl(imoveis["preco"].median()),
    "Preço mediano por m²": brl(imoveis["preco_m2"].median()),
    "Nobre ÷ Periférica (preço/m²)": f"{regioes_m2['Nobre'] / regioes_m2['Periférica']:.1f}×",
    "Tipo com maior preço/m²": f"{tipos_m2.index[0]} ({brl(tipos_m2.iloc[0])})",
    "Segmentos (K-Means) · silhueta": f"{RESULTADOS['k_clusters']} · {RESULTADOS['silhueta']:.2f}",
    "Random Forest (teste): R² · MAPE": f"{RESULTADOS['r2_teste']:.2f} · {RESULTADOS['mape_teste']:.0f}%",
    "Random Forest (5-fold): R²": f"{RESULTADOS['r2_cv_rf']:.2f} ± {RESULTADOS['r2_cv_rf_std']:.2f}",
    "Disponíveis ≥ 20% abaixo do previsto": f"{RESULTADOS['oportunidades']:,}",
    "Registros no Spark (batem com o pandas)": f"{RESULTADOS['registros_spark']:,} ({'sim' if RESULTADOS['registros_spark'] == RESULTADOS['registros_validos'] else 'NÃO'})",
}, name="valor")
display(resumo.to_frame())
display(anual.rename("preço/m² mediano das vendas").to_frame().style.format("R$ {:,.0f}"))

### Principais insights

In [ ]:
trecho_tempo = (f"O preço mediano por m² das vendas varia {variacao_anual:.0%} entre o ano de maior e o de menor valor (com volume mínimo) "
                f"(menor em {anual.idxmin()}, maior em {anual.idxmax()}): "
                + ("não há tendência clara de valorização nem de queda; é preciso mais histórico para falar em sazonalidade."
                   if variacao_anual < 0.10 else "há movimento ao longo do tempo, que vale acompanhar no OLAP (seção 3)."))
outras = 1 - RESULTADOS["imp_area_regiao"]
trecho_venda = ("há diferenças maiores do que o acaso explicaria, que merecem atenção comercial."
                if min(RESULTADOS["p_corretores"], RESULTADOS["p_segmentos"]) < 0.05 else
                "essas diferenças são compatíveis com o acaso, ou seja, não há corretor nem segmento que venda de forma sistematicamente melhor.")
trecho_spark = (f"**Spark e pandas chegam ao mesmo resultado** ({RESULTADOS['registros_spark']:,} registros tratados): a solução já está pronta para crescer em volume sem mudar a lógica."
                if RESULTADOS["registros_spark"] == RESULTADOS["registros_validos"] else
                f"**Atenção:** o Spark tratou {RESULTADOS['registros_spark']:,} registros e o pandas {RESULTADOS['registros_validos']:,}; conferir as regras de limpeza da seção 5.4.")
trecho_kmeans = (f"O K-Means (só com área e preço/m², sem a região) formou {RESULTADOS['k_clusters']} segmentos de tamanho e nível de preço, "
                 f"com NMI de {RESULTADOS['nmi_regiao']:.2f} em relação à região "
                 + ("(reencontrou em boa parte a divisão por região)." if RESULTADOS["nmi_regiao"] >= 0.4 else "(os segmentos separam mais o tamanho do que a região)."))
Markdown(f"""
1. **Localização é o principal determinante do preço.** O m² na região Nobre custa **{RESULTADOS['razao_nobre_periferica']:.1f}×** o da Periférica. {trecho_kmeans}
2. **Dentro da região, o tamanho define o preço.** Área e região somam **{RESULTADOS['imp_area_regiao']:.0%}** da importância no Random Forest (a variável mais importante é `{RESULTADOS['var_top']}`, com {RESULTADOS['var_top_imp']:.0%}).
3. **Os demais atributos pesam pouco:** o tipo de imóvel pesa {RESULTADOS['imp_tipo']:.0%} (em boa parte por carregar a área típica) e quartos, vagas, idade, condomínio e mobília dividem os {outras - RESULTADOS['imp_tipo']:.0%} restantes.
4. **Tempo:** {trecho_tempo}
5. **O modelo serve como preço de referência.** No teste, o Random Forest explica {RESULTADOS['r2_teste']:.0%} da variação do preço, com erro médio de {RESULTADOS['mape_teste']:.0f}% (MAPE); na validação cruzada o R² é {RESULTADOS['r2_cv_rf']:.2f} (± {RESULTADOS['r2_cv_rf_std']:.2f}), contra {RESULTADOS['r2_cv_linear']:.2f} da Regressão Linear. Ele aponta **{RESULTADOS['oportunidades']:,} imóveis disponíveis** anunciados 20% ou mais abaixo do esperado, para revisão.
6. {trecho_spark}
7. **Corretores e segmentos:** a taxa de venda dos {RESULTADOS['n_corretores']} corretores vai de {RESULTADOS['taxa_corretor_min']:.0f}% a {RESULTADOS['taxa_corretor_max']:.0f}% (p = {RESULTADOS['p_corretores']:.2f}) e, entre tipo × região, de {RESULTADOS['seg_pior']} a {RESULTADOS['seg_melhor']} (p = {RESULTADOS['p_segmentos']:.2f}); {trecho_venda}
8. **Idade e condomínio:** o preço do m² varia {RESULTADOS['idade_dif']:+.1%} entre o imóvel mais novo e o mais antigo, descontada a região (Spearman {RESULTADOS['idade_rho']:.2f}). O condomínio pesa {RESULTADOS['cond_peso_baratos']:.2f}% do preço por ano nos imóveis mais baratos e {RESULTADOS['cond_peso_caros']:.2f}% nos mais caros (Spearman entre condomínio e preço: {RESULTADOS['cond_rho']:.2f}).
""")

### Recomendações para a JLeandro Imóveis
* Usar **preço do m² do bairro × área** como ponto de partida na precificação e o modelo como validação.
* **Revisar os anúncios muito abaixo do previsto** (e os muito acima): podem ser erros de cadastro ou boas oportunidades. A lista da seção 4.5 é o ponto de partida.
* **Priorizar a captação nas regiões de maior preço/m²**, onde cada imóvel carrega mais valor.
* Tratar os **segmentos do K-Means** (seção 4.1) como carteiras com metas e divulgação próprias.

### Limitações
* **Base sintética:** os padrões (forte efeito da localização, pouca influência de quartos e vagas, ausência de sazonalidade) e as métricas do modelo refletem as regras do `generator.py`, e não o mercado real.
* **Preço de anúncio, não de fechamento:** não há o valor final negociado.
* **Dados sujos propositais:** cerca de 10% das linhas têm erros inseridos de propósito; as regras de limpeza (seção 2.3) removem os inválidos, mas não corrigem valores plausíveis porém errados.
* **Vendas com data futura** (contadas na seção acima) foram mantidas, pois não há regra de negócio que as invalide.
* **Região por bairro:** vem do mapeamento do próprio generator, não de fonte oficial.
* **Hadoop foi simulado** (blocos, MapReduce e Hive em Python) e o **Spark rodou em modo local**; não houve cluster real.
* O **volume ainda é pequeno** para exigir processamento distribuído; o Spark demonstra como a solução escala (seção 5.10).

### Próximos passos
* Trocar ou complementar a base com **dados reais** (APIs de portais imobiliários) e variáveis do entorno (distância ao metrô, escolas, segurança).
* Acompanhar o modelo em produção e **retreiná-lo** quando os dados mudarem.
* Migrar para **Spark em cluster** (YARN) e ingestão por streaming quando o volume justificar.

### Checklist das etapas do PDF
| Etapa | Onde | Status |
|---|---|---|
| 1. Definição do problema e da base | Seção 1 | ✅ |
| 2. EDA (tipos, qualidade, estatísticas, 9 gráficos interpretados) | Seção 2 | ✅ |
| 3. Armazenamento, ETL e OLAP | Seção 3 | ✅ |
| 4. Data Mining (K-Means) e Machine Learning (Random Forest) | Seção 4 | ✅ |
| 5. Processamento distribuído (Hadoop simulado + Spark) | Seção 5 | ✅ |
| 6. Análise final e insights | Seção 6 | ✅ |